# Host-level Risk Forecasting on the LANL Authentication Stream
## A Temporal Graph Neural Network Tutorial

This notebook builds, trains, and evaluates a **temporal graph neural network (TGNN)** that
forecasts which **host machines** in an enterprise are likely to be **involved in malicious
authentication activity in the near future**, given a stream of authentication events.

We use the LANL Cyber-1 authentication log as the data source and red-team events as a
proxy ground truth for "host involved in malicious activity".

**What you should expect from this notebook:**

- It is written as a **tutorial** for someone who has not used PyTorch or graph neural
  networks before. Every block of code is preceded by a short markdown explanation.
- It is also a **research prototype** — the model, training loop, and evaluation are
  implemented in full, not as placeholders.
- It is **runnable top-to-bottom**, with reproducibility (seeds), device selection
  (CPU/GPU), and a small "sanity-check" pass before the full training run.

**The forecasting task (operational definition we use throughout):**

> At every **decision time** `t` (every 5 minutes), and for every host `i`, predict:
> "Will host `i` appear as either source or destination in at least one **malicious**
> authentication event in the time window `(t, t + 30 minutes]`?"

This is *forecasting*, not anomaly detection. The model never gets to look at the future
when it makes a prediction. We will be very careful about preventing **temporal leakage**.

## 1. Imports and environment setup

Standard scientific Python plus PyTorch for the model. We deliberately keep dependencies
small. In particular, we **do not** use PyTorch Geometric or PyTorch Geometric Temporal,
because the event-driven temporal logic is easier to read when we implement it directly
in PyTorch. We discuss this design choice further in the *Implementation Notes* section.

In [ ]:
import os
import math
import json
import time
import random
import warnings
from collections import defaultdict, deque
from dataclasses import dataclass, field, asdict
from typing import Optional, Tuple, List, Dict, Iterable

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


def set_seed(seed: int) -> None:
    # Make Python, NumPy, and PyTorch deterministic enough for a tutorial.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())

## 2. High-level explanation of the modeling problem

We model authentication logs as a **stream of events**, not as a static graph.

```
raw event log
    -> chronological event stream  e_1, e_2, ..., e_N
    -> per-event host memory updates (event-by-event)
    -> at each fixed decision time t, take a "snapshot":
         - read each host's current memory vector
         - gather its last K events within the trailing W hours
         - summarize those K events with attention
         - combine (memory + summary) -> host embedding
         - pass through a small MLP -> risk probability
```

**Two clocks must be kept clearly separate:**

1. **Continuous event time.** The graph evolves whenever an authentication event occurs.
   Memory updates happen here — irregularly spaced, one per event.
2. **Discrete decision time.** Every 5 minutes, we *score* hosts. Predictions only happen
   here. The model is not asked to predict at every event.

This hybrid (event-driven internal state, periodic external prediction) reflects how a
real defender uses such a system: telemetry arrives continuously, but humans review
risk dashboards on a fixed cadence.

Each event is conceptually:

$$ e_k = (u_k,\; v_k,\; t_k,\; x_k) $$

where `u_k` is source host, `v_k` destination host, `t_k` timestamp, and `x_k` a feature
vector built from authentication metadata and short-term host activity statistics.

## USER CONFIGURATION: UPDATE THESE COLUMN NAMES / FILE PATHS IF NEEDED

All tunable knobs live here. If your column names or paths differ from the standard
LANL Cyber-1 layout, change them in this cell **only**. The rest of the notebook reads
from this `Config` object and never hard-codes paths or names.

### Tutorial / "fast notebook demo" defaults

To make this notebook **actually runnable end-to-end** on a single GPU in a
reasonable amount of time, the defaults below are tuned for a **debug-sized
demo run**, not the full research-spec run. The original spec values are kept
as inline comments so you can scale up once the pipeline is verified.

The reduced defaults are:

- `sample_nrows = 300_000` (down from 2,000,000) — fewer raw events to process.
- `decision_interval_sec = 15 * 60` (up from 5 min) — fewer decision times.
- `K_recent_events = 10` (down from 20) — smaller per-host context window.
- `memory_dim = event_repr_dim = host_emb_dim = attn_hidden_dim = 64`
  (down from 128) — smaller, faster model.
- `replay_encode_chunk_size = 4096` — encode events in chunks instead of one
  giant batch per replay window (lower peak GPU memory, more scalable).

The forecasting time windows (`history_window_sec`, `forecast_horizon_sec`)
match the research design where possible.

**Architecture sizes** (research-spec values are noted in code comments):

- categorical embeddings: 16
- host memory: 64 (spec: 128)
- host embedding (after combine): 64 (spec: 128)
- risk-head hidden: 64
- dropout: 0.1

### Caching flags

We cache the computed per-event numeric features and the per-decision-time
`(active_hosts, labels)` examples to disk under `artifacts_dir`. Reruns load
from cache, which is the single biggest win for iteration speed. Use:

- `force_recompute_features = True` to ignore the cached numeric features.
- `force_recompute_decision_cache = True` to ignore the cached decision
  examples.

In [ ]:
@dataclass
class Config:
    # ---- File paths (USER CONFIG) -----------------------------------------
    data_dir: str = "./data"
    auth_path: str = "../project/data/auth.txt.gz"
    redteam_path: str = "../project/data/redteam.txt.gz"
    figures_dir: str = "./figures"
    artifacts_dir: str = "./artifacts"

    # ---- Column names in the raw auth file (USER CONFIG) ------------------
    # If your CSV uses different headers, edit them here.
    col_time: str = "time"
    col_src_user: str = "src_user"
    col_dst_user: str = "dst_user"
    col_src_host: str = "src_computer"
    col_dst_host: str = "dst_computer"
    col_auth_type: str = "auth_type"
    col_logon_type: str = "logon_type"
    col_auth_orient: str = "auth_orient"
    col_success: str = "success"

    # ---- Column names in the red-team file (USER CONFIG) ------------------
    rt_col_time: str = "time"
    rt_col_user: str = "user"
    rt_col_src_host: str = "src_computer"
    rt_col_dst_host: str = "dst_computer"

    # ---- Loading / subsetting --------------------------------------------
    # The full auth.txt.gz has ~1.05B rows. For one-notebook tractability we
    # load a TIME WINDOW of events anchored on the redteam time range:
    #   - load redteam first
    #   - take t_lo = redteam.time.min() - history_window_sec  (small buffer)
    #   - take t_hi = t_lo + focus_span_sec                    (e.g. 3 days)
    # then stream the auth file in chunks of `load_chunksize` rows and keep
    # only events with t in [t_lo, t_hi], capped at `sample_nrows`.
    #
    # NOTE: sample_nrows is intentionally small for a fast notebook demo.
    # The research-spec runs use 2_000_000 or more rows.
    sample_nrows: Optional[int] = 300_000        # debug demo cap (spec: 2_000_000+)
    focus_span_sec: int = 3 * 86400              # 3 days of auth around redteam
    load_chunksize: int = 1_000_000              # pandas read_csv chunksize

    # ---- Forecasting time windows (seconds) -------------------------------
    # Research spec: W = 6h, H = 30min, decision_interval = 5min.
    # We use a 1h history window and a 15-minute decision interval here so the
    # debug-sized sample yields enough decision times to train on.
    history_window_sec: int = 10 * 60           # W (demo); spec: 6 * 3600
    forecast_horizon_sec: int = 5 * 60          # H = 30 minutes (spec)
    decision_interval_sec: int = 5 * 60         # 15 minutes for fast demo (spec: 5 * 60)

    # ---- Model architecture ----------------------------------------------
    # Demo defaults are smaller and faster. Spec values are 128 across the
    # board. Increase these once the data sample is large enough to justify
    # the extra capacity.
    # K_recent_events: int = 10                    # demo; spec: 20
    # memory_dim: int = 64                         # demo; spec: 128
    cat_emb_dim: int = 16
    # host_emb_dim: int = 64                       # demo; spec: 128
    # risk_hidden_dim: int = 64
    # time_enc_dim: int = 32
    event_repr_dim: int = 64                     # demo; spec: 128
    attn_hidden_dim: int = 64                    # demo; spec: 128
    dropout: float = 0.1

    K_recent_events = 5
    memory_dim = 32
    host_emb_dim = 32
    event_repr_dim = 32
    attn_hidden_dim = 32
    risk_hidden_dim = 32
    time_enc_dim = 16

    # ---- Optimization ----------------------------------------------------
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    num_epochs: int = 10
    early_stop_patience: int = 2
    # How many decision-time prediction batches we accumulate before each
    # gradient step. Truncates BPTT through the memory chain; smaller values
    # save memory but make training noisier.
    decisions_per_grad_step: int = 4

    # ---- Data splits ------------------------------------------------------
    # We split *chronologically* by fractions of the time range present in
    # the loaded sample. A random shuffle would be a temporal-leakage bug.
    train_frac: float = 0.6
    val_frac: float = 0.2
    test_frac: float = 0.2  # implicitly 1 - train - val

    # ---- Speed / memory knobs --------------------------------------------
    # We move event tensors to GPU ONCE and reuse the cached device tensors
    # everywhere instead of repeatedly calling `.to(DEVICE)` inside hot loops.
    # During event replay we encode events in CHUNKS of this size rather than
    # one giant batch covering the whole (prev_dt, dt] window: the per-event
    # GRU memory updates are still applied sequentially (events ordering
    # matters) but the encoder pass is bounded in GPU memory, which is what
    # makes scaling beyond a small sample tractable.
    # replay_encode_chunk_size: int = 4096
    replay_encode_chunk_size: int = 2048

    # ---- Caching flags ---------------------------------------------------
    # Numeric per-event features and the precomputed decision-example cache
    # are saved to disk under artifacts_dir. On rerun we load from disk
    # instead of recomputing (which dominates wall time on warm cache).
    # Set these flags to True to override and recompute.
    force_recompute_features: bool = False
    force_recompute_decision_cache: bool = False

    # ---- Misc ------------------------------------------------------------
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    verbose: bool = True

    # ---- Internal: how many decision times to subsample for sanity pass --
    sanity_max_decisions: int = 8


CFG = Config()
os.makedirs(CFG.figures_dir, exist_ok=True)
os.makedirs(CFG.artifacts_dir, exist_ok=True)
set_seed(CFG.seed)
DEVICE = torch.device(CFG.device)

print("Config:")
print(json.dumps(asdict(CFG), indent=2, default=str))
print("Device:", DEVICE)

## 3. Loading and inspecting the LANL authentication data

The LANL Cyber-1 auth file `auth.txt.gz` is a CSV with no header. The standard schema is:

| col idx | name           |
|---------|----------------|
| 0       | time (seconds) |
| 1       | src_user       |
| 2       | dst_user       |
| 3       | src_computer   |
| 4       | dst_computer   |
| 5       | auth_type      |
| 6       | logon_type     |
| 7       | auth_orient    |
| 8       | success        |

The red-team file `redteam.txt.gz` is also CSV with no header:

| col idx | name           |
|---------|----------------|
| 0       | time (seconds) |
| 1       | user           |
| 2       | src_computer   |
| 3       | dst_computer   |

`?` is used as the missing-value marker.

Our loaders are direct ports of the EDA notebook's loaders (`lanl_cyber1_eda.ipynb`):
the same `pd.read_csv(..., compression="gzip", header=None, names=AUTH_COLS, na_values=["?"])`
contract. We additionally provide `load_auth_focus_window`, which streams the file in
chunks and returns only events whose timestamp falls inside a chosen window.

**Why a time window and not a head sample?** The full `auth.txt.gz` is ~58 days of
events. The first `sample_nrows = 2_000_000` rows only span about *5.7 hours*, but the
red-team activity in `redteam.txt.gz` begins at ~1.74 days and runs through ~30 days.
A naive head sample therefore contains zero positive labels and the supervised model has
nothing to learn. The window loader anchors on the red-team time range so the sample
always overlaps with malicious activity.

In [ ]:
# NOTE: This block is a direct port of the loading pattern in
# `lanl_cyber1_eda.ipynb` (same `pd.read_csv` arguments, same column names,
# same `?` -> NaN handling). The Config column names default to exactly these,
# so the rest of the notebook (which references `CFG.col_*`) keeps working.
#
# We additionally provide `load_auth_focus_window`, which uses chunked
# iteration to slice the auth file by *time* (in seconds since the LANL
# epoch). This is important because the first ~2M rows of auth.txt.gz only
# cover ~5.7 hours, while red-team activity starts at ~1.75 days. A naive
# `nrows=2_000_000` head sample therefore contains ZERO malicious events
# and the supervised model has nothing to learn.

AUTH_COLS = [
    "time", "src_user", "dst_user", "src_computer", "dst_computer",
    "auth_type", "logon_type", "auth_orient", "success",
]
REDTEAM_COLS = ["time", "user", "src_computer", "dst_computer"]


def load_auth_sample(path, nrows=None, chunksize=None, **kwargs):
    # Read auth.txt.gz: `nrows` for a head sample; `chunksize` for a chunked
    # iterator. Mirrors `load_auth_sample` in the EDA notebook.
    kw = dict(
        compression="gzip",
        header=None,
        names=AUTH_COLS,
        na_values=["?"],
        low_memory=False,
    )
    kw.update(kwargs)
    if chunksize:
        return pd.read_csv(path, chunksize=chunksize, **kw)
    return pd.read_csv(path, nrows=nrows, **kw)


def load_redteam(path):
    # Read redteam.txt.gz with the same conventions as the EDA notebook.
    return pd.read_csv(
        path,
        compression="gzip",
        header=None,
        names=REDTEAM_COLS,
        na_values=["?"],
        low_memory=False,
    )


def load_auth_focus_window(
    path: str,
    t_lo: int,
    t_hi: int,
    max_rows: Optional[int] = None,
    chunksize: int = 1_000_000,
    verbose: bool = True,
) -> pd.DataFrame:
    # Stream auth.txt.gz in chunks and keep only events with t_lo <= time <= t_hi.
    # Stops early once we've collected `max_rows` rows or once we move past t_hi
    # (auth.txt.gz is already sorted by time, so once chunk.time.min() > t_hi we
    # can stop reading). This is the recommended loader for the modeling
    # pipeline because it guarantees the sample overlaps red-team activity.
    pieces: List[pd.DataFrame] = []
    kept = 0
    reader = load_auth_sample(path, chunksize=chunksize)
    for i, chunk in enumerate(reader):
        # `time` is integer seconds; convert defensively.
        t = chunk["time"].astype(np.int64)
        if t.iloc[0] > t_hi:
            if verbose:
                print(f"  chunk {i}: starts at t={int(t.iloc[0])} > t_hi={t_hi}, stopping")
            break
        if t.iloc[-1] < t_lo:
            if verbose and (i % 10 == 0):
                print(f"  chunk {i}: ends at t={int(t.iloc[-1])} < t_lo={t_lo}, skipping")
            continue
        sub = chunk[(t >= t_lo) & (t <= t_hi)]
        if len(sub):
            pieces.append(sub)
            kept += len(sub)
            if verbose:
                print(f"  chunk {i}: kept {len(sub):,} rows (running total={kept:,})")
        if max_rows is not None and kept >= max_rows:
            if verbose:
                print(f"  reached max_rows={max_rows:,}, stopping")
            break
    if not pieces:
        raise RuntimeError(
            f"No auth rows found in [t_lo={t_lo}, t_hi={t_hi}]. Check the time range."
        )
    df = pd.concat(pieces, ignore_index=True)
    if max_rows is not None and len(df) > max_rows:
        df = df.iloc[:max_rows].copy()
    return df


def expect_columns(df: pd.DataFrame, expected: List[str], name: str) -> None:
    # Diagnostic helper: raise with a helpful message if columns disagree.
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(
            f"{name} is missing expected columns: {missing}. "
            f"Got: {list(df.columns)}."
        )
    print(f"[{name}] columns OK: {expected}")

We load the auth sample and the red-team file, then run quick diagnostics: shapes,
the first few rows, and unique-value counts on the categorical columns.

In [ ]:
# --- 1. Load redteam (small file, used to anchor the auth time window) ---
print("Loading redteam...")
redteam_raw = load_redteam(CFG.redteam_path)
print("redteam shape:", redteam_raw.shape)
print(redteam_raw.head())
print()

rt_t_min = int(redteam_raw[CFG.rt_col_time].min())
rt_t_max = int(redteam_raw[CFG.rt_col_time].max())
print(f"redteam time range (s): {rt_t_min} -> {rt_t_max} "
      f"({rt_t_min / 86400:.2f}d -> {rt_t_max / 86400:.2f}d since epoch)")

# --- 2. Load auth events around the redteam time range ------------------
# The first ~2M rows of auth.txt.gz only cover ~5.7 hours, but redteam
# activity begins at ~1.74 days. A pure head sample would have 0 positive
# labels. We instead pull a chronological slice of auth that *contains*
# redteam activity, capped at CFG.sample_nrows.
T_LO = max(0, rt_t_min - CFG.history_window_sec)
T_HI = T_LO + CFG.focus_span_sec
print()
print(f"Loading auth events in window [t_lo={T_LO}, t_hi={T_HI}] "
      f"(span {(T_HI - T_LO) / 86400:.2f} days, cap={CFG.sample_nrows:,} rows)...")
auth_raw = load_auth_focus_window(
    CFG.auth_path,
    t_lo=T_LO,
    t_hi=T_HI,
    max_rows=CFG.sample_nrows,
    chunksize=CFG.load_chunksize,
    verbose=True,
)
print()
print("auth shape:", auth_raw.shape)
print(auth_raw.head())
print()

# Sanity diagnostics: downstream cells use CFG.col_* names. With the Config
# defaults those equal AUTH_COLS / REDTEAM_COLS exactly.
expected_auth = [
    CFG.col_time, CFG.col_src_user, CFG.col_dst_user,
    CFG.col_src_host, CFG.col_dst_host,
    CFG.col_auth_type, CFG.col_logon_type, CFG.col_auth_orient, CFG.col_success,
]
expected_rt = [CFG.rt_col_time, CFG.rt_col_user, CFG.rt_col_src_host, CFG.rt_col_dst_host]
expect_columns(auth_raw, expected_auth, "auth")
expect_columns(redteam_raw, expected_rt, "redteam")

# How many redteam events fall inside the auth window we loaded?
rt_in_window = redteam_raw[
    (redteam_raw[CFG.rt_col_time] >= T_LO) & (redteam_raw[CFG.rt_col_time] <= T_HI)
]
print()
print(f"redteam events inside loaded window: {len(rt_in_window)}/{len(redteam_raw)}")
print(f"auth time range loaded (s): {int(auth_raw[CFG.col_time].min())} -> "
      f"{int(auth_raw[CFG.col_time].max())}")
print()
print("Per-column NaN counts (auth):")
print(auth_raw.isna().sum())
print()
print("Unique value counts (small categoricals):")
for c in [CFG.col_auth_type, CFG.col_logon_type, CFG.col_auth_orient, CFG.col_success]:
    if c in auth_raw.columns:
        print(f"  {c}: {auth_raw[c].nunique()} unique")

## 4. Data cleaning and preprocessing

Steps:

1. **Drop rows with missing critical fields** (time, source host, destination host).
   Without these we cannot place the event in the temporal graph.
2. **Sort by time.** The entire downstream pipeline assumes chronological order.
3. **Split user@domain** into separate user and domain fields. We only do this when the
   raw user string contains `@` — otherwise we keep it as-is and use an empty domain.
4. **Fill remaining missing categoricals with `"<UNK>"`** so they get their own embedding.
5. **Mark malicious rows** by joining auth `(time, src_user, src_computer, dst_computer)`
   to redteam `(time, user, src_computer, dst_computer)` (with `user == src_user`).
   This is the same matching used by the LANL benchmark.

The cleaned frame contains exactly the rows we will turn into the event stream. We
preserve string columns at this stage; integer encoding happens in the next section.

In [ ]:
def clean_auth(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    # Drop rows with missing time / src_host / dst_host
    before = len(df)
    df = df.dropna(subset=[cfg.col_time, cfg.col_src_host, cfg.col_dst_host]).copy()
    df[cfg.col_time] = df[cfg.col_time].astype(np.int64)
    df = df.sort_values(cfg.col_time, kind="mergesort").reset_index(drop=True)
    after = len(df)
    print(f"clean_auth: kept {after}/{before} rows after dropping missing critical fields")

    # Split src_user into (src_user_only, src_domain) when '@' is present
    def split_user_domain(series: pd.Series) -> Tuple[pd.Series, pd.Series]:
        s = series.astype(str).fillna("<UNK>")
        parts = s.str.split("@", n=1, expand=True)
        user = parts[0].fillna("<UNK>")
        domain = parts[1] if parts.shape[1] > 1 else pd.Series([""] * len(s))
        domain = domain.fillna("")
        return user, domain

    src_u, src_d = split_user_domain(df[cfg.col_src_user])
    dst_u, dst_d = split_user_domain(df[cfg.col_dst_user])
    df["src_user_only"] = src_u
    df["src_domain"] = src_d
    df["dst_user_only"] = dst_u
    df["dst_domain"] = dst_d

    # Fill remaining missing categoricals with explicit "<UNK>"
    cat_cols = [
        cfg.col_auth_type, cfg.col_logon_type, cfg.col_auth_orient, cfg.col_success,
        "src_user_only", "src_domain",
    ]
    for c in cat_cols:
        df[c] = df[c].astype(str).fillna("<UNK>")
        df.loc[df[c] == "nan", c] = "<UNK>"

    return df


def attach_redteam_label(auth_df: pd.DataFrame, rt_df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    # Mark rows that match a red-team event on (time, src_user, src_host, dst_host).
    rt = rt_df.dropna().copy()
    rt[cfg.rt_col_time] = rt[cfg.rt_col_time].astype(np.int64)
    rt = rt.rename(columns={cfg.rt_col_user: cfg.col_src_user})
    keys = [cfg.col_time, cfg.col_src_user, cfg.col_src_host, cfg.col_dst_host]
    merged = auth_df.merge(
        rt[keys].drop_duplicates(),
        on=keys,
        how="left",
        indicator=True,
    )
    auth_df = auth_df.copy()
    auth_df["is_malicious"] = (merged["_merge"] == "both").to_numpy()
    return auth_df


auth_clean = clean_auth(auth_raw, CFG)
auth_clean = attach_redteam_label(auth_clean, redteam_raw, CFG)
print()
print("auth_clean shape:", auth_clean.shape)
print("malicious rows in sample:", int(auth_clean["is_malicious"].sum()))
print()
print("auth_clean head:")
print(auth_clean.head(3))

## 5. Building the event stream

We turn the cleaned DataFrame into a strict, chronological **event stream**:

- one row per authentication event,
- sorted by timestamp,
- carrying the integer host IDs (next section adds them),
- carrying the `is_malicious` flag for label construction.

We also visualize the temporal volume of events and of malicious events so we can later
discuss train/val/test split boundaries.

In [ ]:
def make_chronological_event_table(auth_df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    df = auth_df[[
        cfg.col_time,
        cfg.col_src_host, cfg.col_dst_host,
        cfg.col_auth_type, cfg.col_logon_type, cfg.col_auth_orient, cfg.col_success,
        "src_user_only", "src_domain",
        "is_malicious",
    ]].copy()
    df = df.sort_values(cfg.col_time, kind="mergesort").reset_index(drop=True)
    return df


events_df = make_chronological_event_table(auth_clean, CFG)
print("events_df shape:", events_df.shape)
print("time range (s):", int(events_df[CFG.col_time].min()), "to", int(events_df[CFG.col_time].max()))
print()
print(events_df.head())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(events_df[CFG.col_time].values // 3600, bins=80, color="steelblue", alpha=0.85)
axes[0].set_xlabel("Hour index"); axes[0].set_ylabel("Events"); axes[0].set_title("Auth event volume by hour (sample)")
mal_t = events_df.loc[events_df["is_malicious"], CFG.col_time].values
if len(mal_t) > 0:
    axes[1].hist(mal_t // 3600, bins=80, color="crimson", alpha=0.85)
axes[1].set_xlabel("Hour index"); axes[1].set_ylabel("Malicious events"); axes[1].set_title("Malicious event volume by hour")
plt.tight_layout()
plt.savefig(os.path.join(CFG.figures_dir, "event_volume.png"), dpi=120)
plt.show()

## 6. Constructing features for each authentication event

For every event we build a feature vector $x_k$ with two flavors of inputs:

**Categorical (each gets its own learned embedding inside the model):**

- `auth_type`
- `logon_type`
- `auth_orient`
- `success` (Success / Failure / unknown)
- `src_user_only` (identity portion of the user@domain string)
- `src_domain`

We **integer-encode** them now and let the model learn `nn.Embedding` tables. Compared to
one-hot, embeddings give us a low-dimensional, learnable representation and handle
high-cardinality fields (users, domains) gracefully.

**Numeric (computed from the past, never the future):**

1. `dt_src_prev`: seconds since the previous event involving the source host
   (as either source or destination).
2. `dt_dst_prev`: same, for the destination host.
3. `roll_src_succ_6h`: count of successful events initiated by the source host in the
   trailing 6h.
4. `roll_src_fail_6h`: count of failed events initiated by the source host in the
   trailing 6h.
5. `roll_src_outdeg_6h`: number of distinct destination hosts the source host contacted
   in the trailing 6h.
6. `roll_dst_indeg_6h`: number of distinct source hosts that contacted the destination
   host in the trailing 6h.

**Why "trailing 6h"?** It matches the history window `W` used at decision time, so the
features summarize the same context the operator would see.

**No future leakage.** Each feature for event `k` uses only events with index `< k`. Below
we implement this with a single chronological pass and per-host bookkeeping.

**Heavy-tailed counts.** Rolling counts span many orders of magnitude. We apply
`log1p` and standardize using **train-only** means and stds further below.

In [ ]:
# ---- 6.1 Integer encode hosts and categorical event fields ----------------

def fit_int_encoder(values: Iterable[str], reserve_unk: bool = True) -> Dict[str, int]:
    """Map each unique value to a stable integer id.

    If `reserve_unk`, id 0 is reserved for `<UNK>` so that unseen values at
    scoring time map to a known slot. We ignore NaNs in the input.
    """
    uniq = sorted({str(v) for v in values if pd.notna(v)})
    enc: Dict[str, int] = {}
    next_id = 0
    if reserve_unk:
        enc["<UNK>"] = 0
        next_id = 1
    for v in uniq:
        if reserve_unk and v == "<UNK>":
            continue
        enc[v] = next_id
        next_id += 1
    return enc


def apply_int_encoder(values: Iterable[str], enc: Dict[str, int]) -> np.ndarray:
    # Map values through the encoder, defaulting to 0 (UNK) for unseen.
    return np.fromiter((enc.get(str(v), 0) for v in values), dtype=np.int64, count=len(values))


# Build host vocab from BOTH src and dst columns so a host has the same id either way.
all_hosts = pd.unique(pd.concat([
    events_df[CFG.col_src_host].astype(str),
    events_df[CFG.col_dst_host].astype(str),
], ignore_index=True))
host2id: Dict[str, int] = {h: i for i, h in enumerate(sorted(all_hosts))}
id2host: Dict[int, str] = {i: h for h, i in host2id.items()}
N_HOSTS = len(host2id)

# Categorical fields and their encoders (fit on the loaded sample only;
# UNK is reserved for runtime-unseen values).
CAT_FIELDS = [
    CFG.col_auth_type,
    CFG.col_logon_type,
    CFG.col_auth_orient,
    CFG.col_success,
    "src_user_only",
    "src_domain",
]
cat_encoders: Dict[str, Dict[str, int]] = {
    f: fit_int_encoder(events_df[f].astype(str).unique()) for f in CAT_FIELDS
}
cat_vocab_sizes: Dict[str, int] = {f: max(cat_encoders[f].values()) + 1 for f in CAT_FIELDS}

print("N_HOSTS:", N_HOSTS)
print("Categorical vocab sizes:", cat_vocab_sizes)

In [ ]:
# ---- 6.2 Numeric features: time gaps + rolling W-second counts ------------
# We compute features in a SINGLE chronological pass to guarantee no leakage.
# For the rolling W-second windows we maintain per-host deques of timestamps
# and pop from the left whenever the head is older than W.
#
# Refactor note: the *previous* version of this cell had two implementations
# of this function (a fast version and a much slower legacy version) plus a
# duplicated `tqdm` import. The slow version has been removed and the imports
# consolidated. This cell now also CACHES the result to a parquet file under
# `CFG.artifacts_dir` so reruns are instantaneous. Set
# `CFG.force_recompute_features = True` to ignore the cache and recompute.

from collections import defaultdict, deque
from tqdm import tqdm

W_SEC = CFG.history_window_sec


def compute_event_features_fast(events_df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """One-pass per-event feature computation.

    For every event k we record:
      - time gaps since the host's previous appearance (src and dst),
      - rolling W-second counts of successful and failed events as source,
      - rolling W-second unique-counterpart out-degree (src) and in-degree (dst).

    We use per-host deques + per-counterpart count maps so the rolling unique
    counterpart count is O(1) amortized rather than O(K) per event.
    """
    n = len(events_df)
    t_arr_local = events_df[cfg.col_time].to_numpy(dtype=np.int64)
    src_arr_local = apply_int_encoder(events_df[cfg.col_src_host].astype(str), host2id)
    dst_arr_local = apply_int_encoder(events_df[cfg.col_dst_host].astype(str), host2id)
    succ_str = events_df[cfg.col_success].astype(str).str.lower().to_numpy()
    is_success_event = succ_str == "success"

    last_seen = np.full(N_HOSTS, -1, dtype=np.int64)
    src_succ_buf = defaultdict(deque)
    src_fail_buf = defaultdict(deque)
    src_outdeg_buf = defaultdict(deque)   # (time, counterpart)
    dst_indeg_buf = defaultdict(deque)    # (time, counterpart)
    src_outdeg_counts = defaultdict(lambda: defaultdict(int))
    dst_indeg_counts = defaultdict(lambda: defaultdict(int))
    src_outdeg_unique = defaultdict(int)
    dst_indeg_unique = defaultdict(int)

    dt_src_prev = np.zeros(n, dtype=np.float32)
    dt_dst_prev = np.zeros(n, dtype=np.float32)
    roll_src_succ = np.zeros(n, dtype=np.float32)
    roll_src_fail = np.zeros(n, dtype=np.float32)
    roll_src_outdeg = np.zeros(n, dtype=np.float32)
    roll_dst_indeg = np.zeros(n, dtype=np.float32)

    SENTINEL_GAP = float(W_SEC * 4)

    for k in tqdm(range(n), desc="compute_event_features_fast"):
        t = int(t_arr_local[k])
        s = int(src_arr_local[k])
        d = int(dst_arr_local[k])

        dt_src_prev[k] = SENTINEL_GAP if last_seen[s] < 0 else (t - last_seen[s])
        dt_dst_prev[k] = SENTINEL_GAP if last_seen[d] < 0 else (t - last_seen[d])

        cutoff = t - W_SEC

        b = src_succ_buf[s]
        while b and b[0] < cutoff:
            b.popleft()
        b = src_fail_buf[s]
        while b and b[0] < cutoff:
            b.popleft()

        b = src_outdeg_buf[s]
        counts = src_outdeg_counts[s]
        while b and b[0][0] < cutoff:
            _, old_dst = b.popleft()
            counts[old_dst] -= 1
            if counts[old_dst] == 0:
                del counts[old_dst]
                src_outdeg_unique[s] -= 1

        b = dst_indeg_buf[d]
        counts = dst_indeg_counts[d]
        while b and b[0][0] < cutoff:
            _, old_src = b.popleft()
            counts[old_src] -= 1
            if counts[old_src] == 0:
                del counts[old_src]
                dst_indeg_unique[d] -= 1

        roll_src_succ[k] = len(src_succ_buf[s])
        roll_src_fail[k] = len(src_fail_buf[s])
        roll_src_outdeg[k] = src_outdeg_unique[s]
        roll_dst_indeg[k] = dst_indeg_unique[d]

        if is_success_event[k]:
            src_succ_buf[s].append(t)
        else:
            src_fail_buf[s].append(t)

        src_outdeg_buf[s].append((t, d))
        if src_outdeg_counts[s][d] == 0:
            src_outdeg_unique[s] += 1
        src_outdeg_counts[s][d] += 1

        dst_indeg_buf[d].append((t, s))
        if dst_indeg_counts[d][s] == 0:
            dst_indeg_unique[d] += 1
        dst_indeg_counts[d][s] += 1

        last_seen[s] = t
        last_seen[d] = t

    return pd.DataFrame({
        "dt_src_prev": np.log1p(dt_src_prev),
        "dt_dst_prev": np.log1p(dt_dst_prev),
        "roll_src_succ_6h": np.log1p(roll_src_succ),
        "roll_src_fail_6h": np.log1p(roll_src_fail),
        "roll_src_outdeg_6h": np.log1p(roll_src_outdeg),
        "roll_dst_indeg_6h": np.log1p(roll_dst_indeg),
    })


def _numeric_features_cache_path(cfg: Config, n_events: int) -> str:
    # Filename includes the parameters that affect the computation so that a
    # change in W or the loaded sample size invalidates the cache automatically.
    fname = f"numeric_features__nrows{n_events}_W{cfg.history_window_sec}.parquet"
    return os.path.join(cfg.artifacts_dir, fname)


def load_or_compute_numeric_features(events_df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Load cached per-event features from parquet if available, else compute and save.

    The cache key embeds n_events and W in the filename so it auto-invalidates
    when the loaded sample or rolling-window length changes.
    """
    cache_path = _numeric_features_cache_path(cfg, len(events_df))
    if (not cfg.force_recompute_features) and os.path.exists(cache_path):
        feats = pd.read_parquet(cache_path)
        if len(feats) == len(events_df):
            print(f"[numeric_features] loaded from cache: {cache_path}")
            return feats
        print(f"[numeric_features] cache size mismatch, recomputing: {cache_path}")
    print(f"[numeric_features] computing fresh ({len(events_df):,} events)...")
    feats = compute_event_features_fast(events_df, cfg)
    feats.to_parquet(cache_path)
    print(f"[numeric_features] saved to: {cache_path}")
    return feats


numeric_features = load_or_compute_numeric_features(events_df, CFG)
print("numeric_features shape:", numeric_features.shape)
print(numeric_features.describe())

In [ ]:
# ---- 6.3 Encode categoricals into integer arrays --------------------------
cat_int = {
    f: apply_int_encoder(events_df[f].astype(str), cat_encoders[f])
    for f in CAT_FIELDS
}
src_id_arr = apply_int_encoder(events_df[CFG.col_src_host].astype(str), host2id)
dst_id_arr = apply_int_encoder(events_df[CFG.col_dst_host].astype(str), host2id)
t_arr = events_df[CFG.col_time].to_numpy(dtype=np.int64)
malicious_arr = events_df["is_malicious"].to_numpy(dtype=bool)

print("src_id_arr dtype/shape:", src_id_arr.dtype, src_id_arr.shape)
print("Per-categorical example head:")
for f in CAT_FIELDS:
    print(f, cat_int[f][:5])

### Visualizations of the engineered features

Quick sanity plots: the histogram of inter-event time gaps (log-seconds) should be
heavy-tailed, and the rolling counts should already span several orders of magnitude
*after* `log1p`. These plots help confirm we want a model with non-linear capacity
(neural network) rather than a small linear classifier.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, numeric_features.columns):
    ax.hist(numeric_features[col].values, bins=60, color="steelblue", alpha=0.85)
    ax.set_title(col)
plt.tight_layout()
plt.savefig(os.path.join(CFG.figures_dir, "numeric_feature_hists.png"), dpi=120)
plt.show()

## 7. Defining the forecasting task with fixed decision times

We discretize the timeline into **decision times** every 5 minutes. At each decision
time `t`:

- **Inputs the model may use:** any event with timestamp `<= t`.
- **Eligible hosts to score:** hosts that were active in the **history window** `(t-W, t]`.
  This is a deliberate simplification: scoring every host in the enterprise at every `t`
  is wasteful because the vast majority have no recent events. Inactive hosts could be
  included for completeness — that is a flag in the training-set builder.
- **Label `y_i(t)`:** 1 iff host `i` appears as source or destination in at least one
  **malicious** authentication event in `(t, t + H]`, else 0.

This is an **operational proxy**: a host that touches a known-malicious event in the next
30 minutes is "involved". It is not a perfect ground truth for "compromised", but it
captures near-term operational risk. We state this clearly because labels strongly
influence whether the model is useful.

The implementation:

1. Build `decision_times` covering only times where the trailing window has data and the
   forecast horizon does not run past the end of the sample.
2. For each decision time, find active hosts using a binary-search lookup on
   chronologically-sorted host appearance times.
3. For labels, precompute the set of `(host_id, t_event)` pairs where that host appears
   in a malicious event. Then for each `(host, decision_time)`, check whether such an
   event exists in `(t, t + H]` (also via binary search).

This avoids materializing the dense `N_hosts x N_decisions` Cartesian product, which can
explode for any reasonable enterprise size.

In [ ]:
# ---- 7.1 Decision-time grid ----------------------------------------------
T_MIN = int(events_df[CFG.col_time].min())
T_MAX = int(events_df[CFG.col_time].max())
DEC_STEP = CFG.decision_interval_sec
H_SEC = CFG.forecast_horizon_sec

dec_start = T_MIN + W_SEC
dec_end = T_MAX - H_SEC
decision_times = np.arange(dec_start, dec_end + 1, DEC_STEP, dtype=np.int64)
print(f"Decision times: {len(decision_times)} (every {DEC_STEP}s, "
      f"from {dec_start} to {dec_end})")

if len(decision_times) < 3:
    raise RuntimeError(
        "Loaded sample is too short for the configured (W, H, decision_interval). "
        f"Span = {T_MAX - T_MIN}s, W = {W_SEC}s, H = {H_SEC}s. "
        "Either increase Config.sample_nrows or shrink Config.history_window_sec."
    )

In [ ]:
# ---- 7.2 Per-host index of involvement times -----------------------------
# For each host, store the sorted timestamps when it appeared (as src OR dst).
host_appearance_times: List[np.ndarray] = [None] * N_HOSTS
host_malicious_times: List[np.ndarray] = [None] * N_HOSTS

# Build via a single pass.
appear_lists = [[] for _ in range(N_HOSTS)]
mal_lists = [[] for _ in range(N_HOSTS)]
for k in range(len(t_arr)):
    s = int(src_id_arr[k]); d = int(dst_id_arr[k]); t = int(t_arr[k])
    appear_lists[s].append(t)
    if d != s:
        appear_lists[d].append(t)
    if malicious_arr[k]:
        mal_lists[s].append(t)
        if d != s:
            mal_lists[d].append(t)
for h in range(N_HOSTS):
    host_appearance_times[h] = np.array(sorted(appear_lists[h]), dtype=np.int64)
    host_malicious_times[h] = np.array(sorted(mal_lists[h]), dtype=np.int64)

n_active_hosts_with_mal = sum(1 for a in host_malicious_times if len(a) > 0)
print("Hosts ever active in sample:",
      sum(1 for a in host_appearance_times if len(a) > 0))
print("Hosts ever in a malicious event:", n_active_hosts_with_mal)

In [ ]:
# ---- 7.3 Helpers: active hosts in a window, and host label for (h, t) -----
#
# Refactor note: `active_hosts_in_window` scans every host on every call,
# binary-searching its appearance times. This is acceptable as a one-time
# preprocessing step but used to be called *inside the training loop* on
# every epoch and every decision time, which made each epoch O(epochs * D *
# N_hosts * log(events_per_host)). We now call it ONCE during preprocessing
# (see `precompute_decision_examples` below) and never again inside hot loops.

def active_hosts_in_window(t: int, w: int) -> np.ndarray:
    """Hosts with at least one appearance in (t - w, t]."""
    ids = []
    lo = t - w
    for h in range(N_HOSTS):
        arr = host_appearance_times[h]
        if arr.size == 0:
            continue
        i_lo = np.searchsorted(arr, lo, side="right")
        i_hi = np.searchsorted(arr, t, side="right")
        if i_hi > i_lo:
            ids.append(h)
    return np.array(ids, dtype=np.int64)


def host_label_at(t: int, h: int, horizon: int) -> int:
    """1 if host h appears in a malicious event in (t, t + horizon], else 0."""
    arr = host_malicious_times[h]
    if arr.size == 0:
        return 0
    i_lo = np.searchsorted(arr, t, side="right")          # strictly after t
    i_hi = np.searchsorted(arr, t + horizon, side="right")
    return int(i_hi > i_lo)


def precompute_decision_examples(
    decision_times_arr: np.ndarray,
    cfg: Config,
) -> Dict[int, Tuple[np.ndarray, np.ndarray]]:
    """Precompute (active hosts, labels) for every decision time, ONCE.

    Returns a dict mapping decision time (int seconds) -> (hosts, labels)
    where `hosts` is a NumPy int64 array of host ids and `labels` is a
    NumPy int8 array of 0/1 values.

    This replaces the old per-epoch pattern of calling
    `active_hosts_in_window` + `host_label_at` inside the training loop.
    The training and evaluation code reads from the returned cache directly,
    so the per-host scanning cost is paid exactly once per process (and
    optionally amortized across runs via `_decision_cache_path`).
    """
    out: Dict[int, Tuple[np.ndarray, np.ndarray]] = {}
    for dt in tqdm(decision_times_arr, desc="precompute_decision_examples"):
        dt_i = int(dt)
        hosts = active_hosts_in_window(dt_i, cfg.history_window_sec)
        if hosts.size == 0:
            continue
        labels = np.fromiter(
            (host_label_at(dt_i, int(h), cfg.forecast_horizon_sec) for h in hosts),
            dtype=np.int8, count=len(hosts),
        )
        out[dt_i] = (hosts, labels)
    return out


def _decision_cache_path(cfg: Config, n_events: int, n_decisions: int) -> str:
    # Cache key embeds the parameters that affect the computation. If any of
    # these change, a new cache file is used and the old one is ignored.
    fname = (
        f"decision_examples__nrows{n_events}"
        f"_W{cfg.history_window_sec}"
        f"_H{cfg.forecast_horizon_sec}"
        f"_step{cfg.decision_interval_sec}"
        f"_nd{n_decisions}.pkl"
    )
    return os.path.join(cfg.artifacts_dir, fname)


def load_or_compute_decision_cache(
    decision_times_arr: np.ndarray,
    cfg: Config,
    n_events: int,
) -> Dict[int, Tuple[np.ndarray, np.ndarray]]:
    """Disk-cached wrapper around `precompute_decision_examples`."""
    import pickle
    cache_path = _decision_cache_path(cfg, n_events, len(decision_times_arr))
    if (not cfg.force_recompute_decision_cache) and os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            cache = pickle.load(f)
        print(f"[decision_examples] loaded from cache: {cache_path} "
              f"({len(cache)} non-empty decision times)")
        return cache
    print(f"[decision_examples] computing fresh ({len(decision_times_arr)} dts)...")
    cache = precompute_decision_examples(decision_times_arr, cfg)
    with open(cache_path, "wb") as f:
        pickle.dump(cache, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"[decision_examples] saved to: {cache_path}")
    return cache


print("Helpers ready.")

## 8. Building the temporal graph data structures

Two concrete tensors and one Python-level cache:

1. **Event tensors.** All event features sit in tensors indexed by event index `k`:
   - `event_t`: timestamps (int64)
   - `event_src`, `event_dst`: integer host ids
   - `event_cat[f]`: integer-encoded categorical fields
   - `event_num`: stacked numeric features (after standardization with train-only stats)
   - `event_malicious`: bool labels

2. **Host memory matrix `mem_table`.** Shape `[N_HOSTS, memory_dim]`. Initialized to zeros
   at the start of training. **Updated in-place over the chronological event stream** by
   the model's `HostMemoryModule`. We discuss its gradient handling explicitly in the
   training section — we use truncated BPTT.

3. **Per-host history cache `host_history`.** For each host `h`, a deque of up to `K`
   recent involvement records. Each record stores:
   - `event_idx`: pointer into the event tensors (so we re-encode features at decision
     time using the current encoder, which keeps the encoder differentiable for context
     events too)
   - `t_event`: timestamp
   - `counterpart_mem_snapshot`: the **other** endpoint's pre-event memory at the time
     of the event, detached. This snapshot is the practical approximation we use for
     "counterpart memory at the time of that historical event" — see *Implementation
     Notes* for why exact snapshots would otherwise be expensive.

Train-only normalization is applied below. `roll_*` and `dt_*_prev` columns are
log1p-transformed already; we now subtract train mean and divide by train std.

In [ ]:
# ---- 8.1 Chronological train/val/test split (NO RANDOM SHUFFLE) -----------
T_RANGE = T_MAX - T_MIN
TRAIN_END_T = int(T_MIN + CFG.train_frac * T_RANGE)
VAL_END_T = int(T_MIN + (CFG.train_frac + CFG.val_frac) * T_RANGE)

train_event_mask = t_arr <= TRAIN_END_T
val_event_mask = (t_arr > TRAIN_END_T) & (t_arr <= VAL_END_T)
test_event_mask = t_arr > VAL_END_T

train_dec_mask = (decision_times <= TRAIN_END_T)
val_dec_mask = (decision_times > TRAIN_END_T) & (decision_times <= VAL_END_T)
test_dec_mask = decision_times > VAL_END_T

print("Train events:", int(train_event_mask.sum()), "| Val:", int(val_event_mask.sum()),
      "| Test:", int(test_event_mask.sum()))
print("Train decision times:", int(train_dec_mask.sum()),
      "| Val:", int(val_dec_mask.sum()), "| Test:", int(test_dec_mask.sum()))
print("Boundaries (s):", TRAIN_END_T, VAL_END_T)

### One-time precomputation of decision-time examples

The training and evaluation loops need, for every decision time `dt`, the set
of currently active hosts and their forecasting labels. The original code
recomputed both **inside the training loop, every epoch and every split**.
Each such call scanned all hosts and binary-searched their appearance times,
which dominates wall time at modest scale.

We move this work **out of the training loop and into one-time preprocessing**:

1. Compute `all_examples = precompute_decision_examples(decision_times, CFG)`
   once, with disk caching, so repeat runs are instant.
2. Slice it by the chronological split masks to get `train_examples`,
   `val_examples`, `test_examples` (each a dict of `dt -> (hosts, labels)`).

After this cell, `active_hosts_in_window` and `host_label_at` are **never**
called inside training/evaluation loops again — those loops just look up the
precomputed cache by `dt`.

In [ ]:
# ---- 8.1b One-time precompute of decision-time (hosts, labels) cache ------
# This is the single biggest speedup vs. the previous notebook: instead of
# scanning every host on every decision time on every epoch, we compute the
# full {dt: (hosts, labels)} dictionary ONCE (with on-disk caching) and slice
# it by the train/val/test masks. The hot training loop then only does a
# dict lookup per decision time.

DEC_TRAIN = decision_times[train_dec_mask]
DEC_VAL = decision_times[val_dec_mask]
DEC_TEST = decision_times[test_dec_mask]

all_examples = load_or_compute_decision_cache(decision_times, CFG, n_events=len(t_arr))

train_examples: Dict[int, Tuple[np.ndarray, np.ndarray]] = {
    int(dt): all_examples[int(dt)] for dt in DEC_TRAIN if int(dt) in all_examples
}
val_examples: Dict[int, Tuple[np.ndarray, np.ndarray]] = {
    int(dt): all_examples[int(dt)] for dt in DEC_VAL if int(dt) in all_examples
}
test_examples: Dict[int, Tuple[np.ndarray, np.ndarray]] = {
    int(dt): all_examples[int(dt)] for dt in DEC_TEST if int(dt) in all_examples
}


def _split_stats(name: str, ex: Dict[int, Tuple[np.ndarray, np.ndarray]]) -> None:
    n_dts = len(ex)
    n_examples = sum(len(h) for h, _ in ex.values())
    n_pos = sum(int(l.sum()) for _, l in ex.values())
    print(f"  {name:5s}: {n_dts:5d} decision times, {n_examples:8d} (host, dt) "
          f"examples, {n_pos:6d} positives")


print(f"Decision-example cache populated. Per-split sizes:")
_split_stats("train", train_examples)
_split_stats("val", val_examples)
_split_stats("test", test_examples)

In [ ]:
# ---- 8.2 Standardize numeric features using TRAIN-ONLY statistics ---------
num_train = numeric_features.loc[train_event_mask]
num_mean = num_train.mean().to_numpy()
num_std = num_train.std().replace(0, 1.0).to_numpy()
print("Train numeric means:", num_mean)
print("Train numeric stds :", num_std)

num_norm = (numeric_features.to_numpy() - num_mean) / num_std

In [ ]:
# ---- 8.3 Move event tensors to torch ---------------------------------------
#
# Refactor note: the previous notebook materialized CPU tensors here and then
# repeatedly called `event_num.to(DEVICE)[idx]` and
# `event_cat[f].to(DEVICE)[idx]` inside `replay_through_time` and
# `build_context_batch` — that is, it copied the entire E-by-F tensor across
# the PCIe bus on every call. With ~2M events that is by far the largest
# wall-time cost of training.
#
# We now create *persistent device-side copies* of the per-event categorical
# and numeric tensors ONCE, right next to the CPU tensors. The downstream
# code reads from these device tensors directly (`event_cat_dev[f][idx_dev]`
# / `event_num_dev[idx_dev]`), so each replay step does exactly one tiny
# index gather on the GPU and zero host->device transfers.
#
# We deliberately keep `event_t`, `event_src`, `event_dst`, `event_malicious`
# on CPU because they are used for lightweight Python bookkeeping
# (`event_t[idx_cpu].numpy()`, last-seen lookups, label construction) and
# never participate in model computation. Mixing CPU/GPU here is intentional:
# CPU indices for CPU tensors, GPU indices for GPU tensors.

# CPU tensors (used for indexing / Python bookkeeping)
event_t = torch.from_numpy(t_arr.astype(np.int64))                                  # [E]
event_src = torch.from_numpy(src_id_arr.astype(np.int64))                          # [E]
event_dst = torch.from_numpy(dst_id_arr.astype(np.int64))                          # [E]
event_num = torch.from_numpy(num_norm.astype(np.float32))                          # [E, F_num]
event_cat = {f: torch.from_numpy(cat_int[f].astype(np.int64)) for f in CAT_FIELDS}  # each [E]
event_malicious = torch.from_numpy(malicious_arr.astype(np.bool_))                 # [E]

# Persistent device-side copies (used for ALL model inputs in the hot loop)
# WARNING: these tensors live on the GPU for the entire run. With the demo
# defaults their footprint is small (a few hundred MB). For very large
# samples you may want to keep them on CPU and accept the per-call transfer.
event_num_dev = event_num.to(DEVICE)
event_cat_dev = {f: event_cat[f].to(DEVICE) for f in CAT_FIELDS}

print("event_t shape:", event_t.shape, "dtype:", event_t.dtype)
print("event_src shape:", event_src.shape)
print("event_num shape:", event_num.shape, "device:", event_num.device,
      "| event_num_dev device:", event_num_dev.device)
for f in CAT_FIELDS:
    print(f"event_cat[{f!r}] shape:", event_cat[f].shape,
          "| dev:", event_cat_dev[f].device)
print("event_malicious shape:", event_malicious.shape)

## 9. Implementing the temporal host-memory model

We now implement each model component as its own `nn.Module`. The modules are:

1. `TimeEncoding` — sinusoidal encoding of a continuous time gap (TGN-style).
2. `EventFeatureEncoder` — maps categorical ids + numeric features into one event vector.
3. `HostMemoryModule` — for each event, computes per-endpoint messages and applies
   `nn.GRUCell` to update memory.
4. `TemporalAttention` — over the last `K` events of a host, attention with the host's
   current memory as query.
5. `RiskHead` — `Linear(128 -> 64) -> ReLU -> Dropout(0.1) -> Linear(64 -> 1)`.
6. `TemporalHostRiskModel` — wires the above together.

**Why GRUCell, not full GRU?** The events arrive irregularly and we update one event at a
time, with the time gap encoded explicitly. A full `nn.GRU` would require us to batch a
fixed-length sequence, which does not match how the temporal graph actually evolves.

In [ ]:
class TimeEncoding(nn.Module):
    # TGN-style learnable sinusoidal time encoding.
    # Input:  dt of shape [...]   (seconds since some reference)
    # Output: [..., dim]
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        # Log-spaced initial frequencies covering ~1s to ~1e9s
        freqs = 1.0 / (10.0 ** np.linspace(0, 9, dim))
        self.basis_freq = nn.Parameter(torch.from_numpy(freqs).float())
        self.phase = nn.Parameter(torch.zeros(dim))

    def forward(self, dt: torch.Tensor) -> torch.Tensor:
        # dt can be any shape; we add a trailing dim for broadcasting.
        x = dt.float().unsqueeze(-1) * self.basis_freq + self.phase
        return torch.cos(x)

In [ ]:
class EventFeatureEncoder(nn.Module):
    # Build a single event vector from cat ids and numeric features.
    # Input shapes (for a batch of B events):
    #   cat_ids: dict {field_name: LongTensor[B]}
    #   num:     FloatTensor[B, F_num]
    # Output shape: FloatTensor[B, out_dim]
    def __init__(self, cat_vocab: Dict[str, int], cat_emb_dim: int,
                 num_numeric: int, hidden: int, out_dim: int):
        super().__init__()
        self.cat_names = list(cat_vocab.keys())
        self.embeddings = nn.ModuleDict({
            n: nn.Embedding(num_embeddings=v, embedding_dim=cat_emb_dim)
            for n, v in cat_vocab.items()
        })
        in_dim = cat_emb_dim * len(self.cat_names) + num_numeric
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, cat_ids: Dict[str, torch.Tensor], num: torch.Tensor) -> torch.Tensor:
        embs = [self.embeddings[n](cat_ids[n]) for n in self.cat_names]
        x = torch.cat(embs + [num], dim=-1)
        return self.mlp(x)

In [ ]:
class HostMemoryModule(nn.Module):
    """Per-event message + GRU update of the host memory table.

    Refactor note: the global host memory table (`mem_table`) is now kept on
    CPU. It is a mutable state store of shape `[N_HOSTS, mem_dim]` rather
    than a dense GPU batch tensor used in linear algebra, so storing it on
    GPU is a poor use of (precious, scarce) device memory. For each event
    we move only the two endpoint rows to the device for the message + GRU
    computation, then move the (detached) updated rows back to the CPU
    table. The `(src_pre, dst_pre)` snapshots saved into the history deque
    are also stored on CPU because they are pure data, not parameters.

    The dead `update_one` variants from the previous notebook (one with a
    full table clone, one with the autograd-unsafe in-place GPU write)
    have been removed.
    """

    def __init__(self, mem_dim: int, event_dim: int, time_dim: int):
        super().__init__()
        self.mem_dim = mem_dim
        self.time_enc = TimeEncoding(time_dim)
        msg_in = mem_dim * 2 + event_dim + time_dim
        self.msg_mlp = nn.Sequential(
            nn.Linear(msg_in, mem_dim),
            nn.ReLU(),
            nn.Linear(mem_dim, mem_dim),
        )
        self.gru = nn.GRUCell(input_size=mem_dim, hidden_size=mem_dim)

    def message(self, self_mem: torch.Tensor, other_mem: torch.Tensor,
                event_feat: torch.Tensor, dt_self: torch.Tensor) -> torch.Tensor:
        # All shapes: [B, *]. Inputs are expected to live on the same device
        # (typically the GPU; this is called from the per-event update below).
        te = self.time_enc(dt_self)
        return self.msg_mlp(torch.cat([self_mem, other_mem, event_feat, te], dim=-1))

    def update_one(self,
                   mem_table_cpu: torch.Tensor,
                   src_id: int, dst_id: int,
                   event_feat_dev: torch.Tensor,
                   dt_src_dev: torch.Tensor, dt_dst_dev: torch.Tensor,
                   device: torch.device,
                   ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Apply one event's update to the CPU memory table.

        Inputs:
          mem_table_cpu: [N_HOSTS, mem_dim] CPU float tensor (mutated in place)
          src_id, dst_id: int host ids
          event_feat_dev: [1, event_dim] on `device`
          dt_src_dev, dt_dst_dev: [1] on `device`, time gaps for the message
          device: the device on which the per-event neural computation runs

        Returns:
          mem_table_cpu (same object, with two rows updated)
          src_pre_cpu:  [mem_dim] CPU snapshot of src memory BEFORE the update
          dst_pre_cpu:  [mem_dim] CPU snapshot of dst memory BEFORE the update

        We deliberately move only two `mem_dim`-sized vectors per event to
        the GPU (instead of indexing into a GPU-resident table). On the way
        back we detach + .cpu() before writing into `mem_table_cpu`, so the
        table remains a pure CPU data buffer with no autograd graph.
        """
        # Read CPU rows for the two endpoints (cheap: 2 * mem_dim floats).
        src_pre_cpu = mem_table_cpu[src_id].clone().unsqueeze(0)   # [1, mem_dim] CPU
        dst_pre_cpu = mem_table_cpu[dst_id].clone().unsqueeze(0)   # [1, mem_dim] CPU

        # Move ONLY these two small vectors to the device for the GRU/message.
        src_pre = src_pre_cpu.to(device, non_blocking=True)
        dst_pre = dst_pre_cpu.to(device, non_blocking=True)

        msg_s = self.message(src_pre, dst_pre, event_feat_dev, dt_src_dev)
        msg_d = self.message(dst_pre, src_pre, event_feat_dev, dt_dst_dev)
        new_src = self.gru(msg_s, src_pre)                         # [1, mem_dim] dev
        new_dst = self.gru(msg_d, dst_pre)                         # [1, mem_dim] dev

        # Write back to the CPU table as detached data (no autograd link).
        mem_table_cpu[src_id] = new_src.detach().squeeze(0).cpu()
        mem_table_cpu[dst_id] = new_dst.detach().squeeze(0).cpu()

        # Snapshots returned for the history cache are CPU-resident data.
        return mem_table_cpu, src_pre_cpu.squeeze(0), dst_pre_cpu.squeeze(0)

## 10. Implementing event-based temporal aggregation

For a host `i` at decision time `t`, we collect its **last K events** (within `[t-W, t]`)
from the per-host history cache. For each such event we build a context vector with
three parts:

- the **counterpart's pre-event memory** snapshot (already detached; data, not a
  parameter)
- the **encoded event features** (re-encoded with the *current* `EventFeatureEncoder`
  so gradients flow through the encoder for context events too)
- a sinusoidal encoding of `dt = t - t_event`

We then run multi-head-style scaled dot-product attention with the host's **current
memory** as the query; the attention output is `c_i(t)`. Empty slots (when the host has
fewer than `K` events) are masked out.

Finally we combine `[m_i(t) ; c_i(t)]` with a small MLP into the host embedding
`h_i(t) ∈ R^{128}` that the risk head consumes.

In [ ]:
class TemporalAttention(nn.Module):
    # Single-head scaled dot-product attention over K context vectors.
    # Inputs:
    #   query_mem:        [B, mem_dim]     (host memory used as query)
    #   ctx_counterpart:  [B, K, mem_dim]  (counterpart memory snapshots)
    #   ctx_event_feat:   [B, K, event_dim]
    #   ctx_dt:           [B, K]
    #   mask:             [B, K] bool (True = valid slot)
    # Output:
    #   [B, out_dim]
    def __init__(self, mem_dim: int, event_dim: int, time_dim: int,
                 hidden: int, out_dim: int):
        super().__init__()
        self.time_enc = TimeEncoding(time_dim)
        self.ctx_dim = mem_dim + event_dim + time_dim
        self.q_proj = nn.Linear(mem_dim, hidden)
        self.k_proj = nn.Linear(self.ctx_dim, hidden)
        self.v_proj = nn.Linear(self.ctx_dim, hidden)
        self.out_proj = nn.Linear(hidden, out_dim)
        self.scale = hidden ** 0.5

    def forward(self, query_mem: torch.Tensor,
                ctx_counterpart: torch.Tensor,
                ctx_event_feat: torch.Tensor,
                ctx_dt: torch.Tensor,
                mask: torch.Tensor) -> torch.Tensor:
        te = self.time_enc(ctx_dt)                                  # [B, K, time_dim]
        ctx = torch.cat([ctx_counterpart, ctx_event_feat, te], dim=-1)  # [B, K, ctx_dim]
        Q = self.q_proj(query_mem).unsqueeze(1)                     # [B, 1, H]
        Kp = self.k_proj(ctx)                                       # [B, K, H]
        Vp = self.v_proj(ctx)                                       # [B, K, H]
        scores = (Q * Kp).sum(-1) / self.scale                      # [B, K]

        # Mask invalid slots. If ALL slots are invalid, set scores to 0
        # before softmax to avoid NaNs and zero out the output below.
        all_invalid = (~mask).all(dim=-1, keepdim=True)             # [B, 1]
        scores = scores.masked_fill(~mask, float("-inf"))
        scores = torch.where(all_invalid.expand_as(scores),
                             torch.zeros_like(scores), scores)
        attn = F.softmax(scores, dim=-1)                            # [B, K]
        attn = torch.where(all_invalid.expand_as(attn),
                           torch.zeros_like(attn), attn)
        out = (attn.unsqueeze(-1) * Vp).sum(dim=1)                  # [B, H]
        return self.out_proj(out)                                   # [B, out_dim]

## 11. Building the risk prediction head

Tiny MLP. Input is `h_i(t) ∈ R^{128}`. Output is a single logit, which we sigmoid into
a probability. We use `BCEWithLogitsLoss` for numerical stability, so the head returns
a logit (not yet sigmoid'd) during training; we apply sigmoid in the eval helper.

In [ ]:
class RiskHead(nn.Module):
    def __init__(self, in_dim: int, hidden: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        return self.net(h).squeeze(-1)

In [ ]:
class TemporalHostRiskModel(nn.Module):
    def __init__(self, cat_vocab: Dict[str, int], num_numeric: int, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.event_encoder = EventFeatureEncoder(
            cat_vocab=cat_vocab,
            cat_emb_dim=cfg.cat_emb_dim,
            num_numeric=num_numeric,
            hidden=cfg.event_repr_dim,
            out_dim=cfg.event_repr_dim,
        )
        self.memory_module = HostMemoryModule(
            mem_dim=cfg.memory_dim,
            event_dim=cfg.event_repr_dim,
            time_dim=cfg.time_enc_dim,
        )
        self.attention = TemporalAttention(
            mem_dim=cfg.memory_dim,
            event_dim=cfg.event_repr_dim,
            time_dim=cfg.time_enc_dim,
            hidden=cfg.attn_hidden_dim,
            out_dim=cfg.host_emb_dim,
        )
        self.combine = nn.Sequential(
            nn.Linear(cfg.memory_dim + cfg.host_emb_dim, cfg.host_emb_dim),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
        )
        self.risk_head = RiskHead(cfg.host_emb_dim, cfg.risk_hidden_dim, cfg.dropout)

    # --- Convenience wrappers; the training loop calls these directly ---
    def encode_events(self, cat_ids: Dict[str, torch.Tensor],
                      num: torch.Tensor) -> torch.Tensor:
        return self.event_encoder(cat_ids, num)

    def predict(self, query_mem: torch.Tensor,
                ctx_counterpart: torch.Tensor,
                ctx_event_feat: torch.Tensor,
                ctx_dt: torch.Tensor,
                mask: torch.Tensor) -> torch.Tensor:
        c = self.attention(query_mem, ctx_counterpart, ctx_event_feat, ctx_dt, mask)
        h = self.combine(torch.cat([query_mem, c], dim=-1))
        return self.risk_head(h)

## 12. Creating training examples

Two-phase organization:

- **Phase 1 — done above.** We turned raw logs into chronological event tensors plus
  per-host indices. Decision-time labels can be queried lazily.
- **Phase 2 — described here and run in the next section.** We **replay events** in
  chronological order, updating `mem_table` and per-host history. Whenever a decision
  time is reached, we score that decision time's active hosts and accumulate loss.

The replay-and-score pattern means we never need a giant `(host, t)` Cartesian product
in memory. We score on-the-fly as the simulated clock passes each decision time.

Below we implement one helper, `build_context_batch`, that takes a host and a decision
time and returns the tensors needed for the attention over its `K` recent events. This
is used both during training and during evaluation.

In [ ]:
# host_history[h] is a deque of dicts:
#   {"event_idx": int, "t_event": int, "counterpart_mem": Tensor[mem_dim] CPU}
# All snapshots in the history cache live on CPU. We only move the
# assembled context batch to the device once, right before attention.
host_history: List[deque] = [deque(maxlen=CFG.K_recent_events) for _ in range(N_HOSTS)]


def reset_temporal_state(model: TemporalHostRiskModel, cfg: Config
                         ) -> Tuple[torch.Tensor, List[deque]]:
    """Fresh memory and empty history (call at the start of each epoch).

    Refactor note: `mem_table` is now a CPU tensor. It is a mutable state
    store, not a dense GPU batch matrix used in linear algebra. Keeping it
    on CPU drastically reduces the GPU memory footprint, which is the main
    constraint on training larger models.
    """
    mem_table_cpu = torch.zeros(N_HOSTS, cfg.memory_dim)
    history = [deque(maxlen=cfg.K_recent_events) for _ in range(N_HOSTS)]
    return mem_table_cpu, history


def build_context_batch(model: TemporalHostRiskModel,
                        mem_table_cpu: torch.Tensor,
                        history: List[deque],
                        hosts_cpu: torch.Tensor,
                        dt: int,
                        cfg: Config,
                        device: torch.device,
                        ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor,
                                   torch.Tensor, torch.Tensor]:
    """Build attention inputs for a batch of hosts at decision time `dt`.

    Performance design:
    - `query_mem`, `ctx_counterpart`, `ctx_dt`, and `mask` are first
      assembled on CPU from the host history cache (which is CPU-resident).
      Each one is then moved to the device in **one** transfer instead of
      being filled element-by-element with many tiny GPU writes from the
      Python loop.
    - All referenced event indices are gathered into a single 1-D tensor,
      passed through the *cached* device-side event tensors
      (`event_cat_dev` / `event_num_dev` — no `.to(DEVICE)` on the full
      tensor every call), encoded ONCE with the current
      `EventFeatureEncoder` (so gradients still flow through the encoder
      for context events during training), and scattered back into a
      `[B, K, event_repr_dim]` buffer with a single advanced-indexing
      assignment.

    Inputs:
      hosts_cpu: [B] int64 host ids on CPU (CPU index for the CPU mem_table)
      dt:        decision time in seconds
      device:    the device on which the model lives

    Returns five tensors (all on `device`):
      query_mem       [B, mem_dim]
      ctx_counterpart [B, K, mem_dim]
      ctx_event_feat  [B, K, event_repr_dim]
      ctx_dt          [B, K]
      mask            [B, K]   bool
    """
    B = int(hosts_cpu.shape[0])
    K = cfg.K_recent_events
    mem_dim = cfg.memory_dim

    # ---- 1) CPU staging of the per-host context buffers ------------------
    query_mem_cpu = mem_table_cpu.index_select(0, hosts_cpu)        # [B, mem_dim]
    ctx_counterpart_cpu = torch.zeros(B, K, mem_dim)
    ctx_dt_cpu = torch.zeros(B, K)
    mask_cpu = torch.zeros(B, K, dtype=torch.bool)

    flat_event_idxs: List[int] = []
    flat_b: List[int] = []
    flat_k: List[int] = []

    hosts_list = hosts_cpu.tolist()
    for b in range(B):
        h = hosts_list[b]
        recs = list(history[h])[-K:]
        for k_idx, rec in enumerate(recs):
            ctx_counterpart_cpu[b, k_idx] = rec["counterpart_mem"]
            ctx_dt_cpu[b, k_idx] = float(dt - rec["t_event"])
            mask_cpu[b, k_idx] = True
            flat_event_idxs.append(rec["event_idx"])
            flat_b.append(b)
            flat_k.append(k_idx)

    # ---- 2) Single-shot transfers to device ------------------------------
    query_mem = query_mem_cpu.to(device, non_blocking=True)
    ctx_counterpart = ctx_counterpart_cpu.to(device, non_blocking=True)
    ctx_dt = ctx_dt_cpu.to(device, non_blocking=True)
    mask = mask_cpu.to(device, non_blocking=True)

    # ---- 3) Encode ALL referenced events in one batched pass --------------
    if flat_event_idxs:
        idx_dev = torch.as_tensor(flat_event_idxs, dtype=torch.long, device=device)
        cat_batch = {f: event_cat_dev[f].index_select(0, idx_dev) for f in CAT_FIELDS}
        num_batch = event_num_dev.index_select(0, idx_dev)
        feats = model.encode_events(cat_batch, num_batch)            # [N, event_dim]
        ctx_event_feat = torch.zeros(B, K, feats.size(-1), device=device)
        b_t = torch.as_tensor(flat_b, dtype=torch.long, device=device)
        k_t = torch.as_tensor(flat_k, dtype=torch.long, device=device)
        ctx_event_feat[b_t, k_t] = feats
    else:
        ctx_event_feat = torch.zeros(B, K, cfg.event_repr_dim, device=device)

    return query_mem, ctx_counterpart, ctx_event_feat, ctx_dt, mask

## 13. Training loop

The training loop replays events and predicts at decision times. The key loop body:

```
for each decision time dt in chronological order:
    # 1) Replay all events in (prev_t, dt] in CHUNKS of size
    #    cfg.replay_encode_chunk_size. Per chunk:
    #      - batch-encode all events on the GPU in one forward pass,
    #        reading from the persistent device-side event tensors
    #        (event_cat_dev, event_num_dev) — no per-call .to(DEVICE)
    #      - sequentially apply per-event GRU updates so event ordering
    #        is preserved; only the two endpoint rows of `mem_table_cpu`
    #        round-trip to the GPU for the message + GRU computation
    #      - append the counterpart's pre-update memory snapshot to each
    #        endpoint's per-host history deque (CPU-resident)
    # 2) If active hosts exist at dt (looked up in the precomputed
    #    examples cache, not recomputed):
    #      - build the context batch (CPU-staged, single .to(device))
    #      - run the model, compute weighted BCE loss
    #      - accumulate the loss over `decisions_per_grad_step` decision times
    # 3) Periodically: backward, optimizer step. We do NOT need to
    #    .detach() `mem_table` anymore because it is a CPU data buffer
    #    written via .detach().cpu() inside HostMemoryModule.update_one,
    #    so it carries no autograd graph by construction.
```

**Where do gradients flow?** Into the event encoder via the **context-event**
re-encoding inside `build_context_batch` (these are encoded with grad enabled), and
into the attention layer + combine MLP + risk head from `model.predict`. The CPU
`mem_table` and the per-event GRU update no longer participate in the loss graph;
the per-event GRU acts as a learned online state estimator whose state we observe
through `mem_table`.

**Class imbalance.** Positives are very rare (a small fraction of host-time pairs touch a
malicious event). We compute `pos_weight` for `BCEWithLogitsLoss` from the training
labels (using the precomputed cache, so this is essentially free) and pass it to the
loss function. This re-weights positive examples upward.

In [ ]:
def chronological_event_indices(t_arr_np: np.ndarray, t_lo: int, t_hi: int) -> np.ndarray:
    """Indices of events with t_lo < t <= t_hi (the (prev_dt, dt] window)."""
    i_lo = np.searchsorted(t_arr_np, t_lo, side="right")
    i_hi = np.searchsorted(t_arr_np, t_hi, side="right")
    return np.arange(i_lo, i_hi)


def replay_through_time(model: TemporalHostRiskModel,
                        mem_table_cpu: torch.Tensor,
                        history: List[deque],
                        from_t: int, to_t: int,
                        cfg: Config,
                        update_with_grad: bool,
                        device: torch.device,
                        ) -> torch.Tensor:
    """Process events with `from_t < t <= to_t`, updating CPU memory + history.

    Refactor notes (in approximate order of impact):

    1. The full event window used to be encoded in one giant batched call to
       `model.encode_events`. For long windows this allocated O(window_size *
       event_repr_dim) on the GPU. We now process the window in CHUNKS of
       `cfg.replay_encode_chunk_size`. Encoding is still batched (one call
       per chunk) so the encoder sees real-sized batches, but peak GPU
       memory is bounded by the chunk size.

    2. The per-event GRU memory update remains sequential because consecutive
       events can share a host: the second event must see the first event's
       update. Sequentialism is intrinsic to the modeling assumption — what
       we optimize is the *amount of work per event* and the *device traffic
       per event*, not the loop ordering.

    3. We read from the persistent device-side event tensors (`event_cat_dev`,
       `event_num_dev`) instead of re-uploading the whole CPU event tensors
       on every call.

    4. The per-event time-gap inputs (`dt_src`, `dt_dst`) are pre-computed on
       CPU into NumPy arrays for the whole chunk and uploaded to the device
       in one transfer; the inner loop just slices into those arrays.
    """
    idxs = chronological_event_indices(t_arr, from_t, to_t)
    n_total = len(idxs)
    if n_total == 0:
        return mem_table_cpu

    # CPU bookkeeping for time gaps. We use a single int64 vector to track
    # last-seen timestamps across chunks within this replay call.
    SENT = float(cfg.history_window_sec * 4)
    last_seen_local = np.full(N_HOSTS, -1, dtype=np.int64)

    chunk_size = max(1, int(cfg.replay_encode_chunk_size))

    for chunk_start in range(0, n_total, chunk_size):
        chunk_end = min(chunk_start + chunk_size, n_total)
        chunk_idxs = idxs[chunk_start:chunk_end]
        chunk_idxs_t_cpu = torch.from_numpy(chunk_idxs).long()
        chunk_idxs_t_dev = chunk_idxs_t_cpu.to(device, non_blocking=True)

        # ---- 1) Encode the entire chunk in ONE pass on the device --------
        cat_batch = {f: event_cat_dev[f].index_select(0, chunk_idxs_t_dev)
                     for f in CAT_FIELDS}
        num_batch = event_num_dev.index_select(0, chunk_idxs_t_dev)
        if update_with_grad:
            ev_feats = model.encode_events(cat_batch, num_batch)   # [chunk, evt_dim]
        else:
            with torch.no_grad():
                ev_feats = model.encode_events(cat_batch, num_batch)

        # ---- 2) Pre-compute per-event dt arrays on CPU, then upload ------
        src_local = event_src.index_select(0, chunk_idxs_t_cpu).numpy()
        dst_local = event_dst.index_select(0, chunk_idxs_t_cpu).numpy()
        t_local = event_t.index_select(0, chunk_idxs_t_cpu).numpy()
        n_chunk = len(chunk_idxs)

        dt_s_arr = np.empty(n_chunk, dtype=np.float32)
        dt_d_arr = np.empty(n_chunk, dtype=np.float32)
        for j in range(n_chunk):
            s = int(src_local[j]); d = int(dst_local[j]); t = int(t_local[j])
            prev_s = last_seen_local[s]
            prev_d = last_seen_local[d]
            dt_s_arr[j] = SENT if prev_s < 0 else max(0, t - prev_s)
            dt_d_arr[j] = SENT if prev_d < 0 else max(0, t - prev_d)
            last_seen_local[s] = t
            last_seen_local[d] = t
        dt_s_dev = torch.from_numpy(dt_s_arr).to(device, non_blocking=True)
        dt_d_dev = torch.from_numpy(dt_d_arr).to(device, non_blocking=True)

        # ---- 3) Apply the per-event GRU update sequentially --------------
        # `update_one` reads two CPU rows, moves them to GPU, runs the
        # message + GRU, and writes detached results back to the CPU table.
        for j in range(n_chunk):
            e_idx = int(chunk_idxs[j])
            s = int(src_local[j]); d = int(dst_local[j]); t = int(t_local[j])
            ev_feat = ev_feats[j:j + 1]
            dt_s = dt_s_dev[j:j + 1]
            dt_d = dt_d_dev[j:j + 1]

            if update_with_grad:
                mem_table_cpu, src_pre_cpu, dst_pre_cpu = (
                    model.memory_module.update_one(
                        mem_table_cpu, s, d, ev_feat, dt_s, dt_d, device
                    )
                )
            else:
                with torch.no_grad():
                    mem_table_cpu, src_pre_cpu, dst_pre_cpu = (
                        model.memory_module.update_one(
                            mem_table_cpu, s, d, ev_feat, dt_s, dt_d, device
                        )
                    )

            # History entries: COUNTERPART pre-event memory as CPU data.
            history[s].append({"event_idx": e_idx, "t_event": t,
                               "counterpart_mem": dst_pre_cpu})
            history[d].append({"event_idx": e_idx, "t_event": t,
                               "counterpart_mem": src_pre_cpu})

    return mem_table_cpu

In [ ]:
from tqdm.auto import tqdm


def compute_pos_weight_from_split(
    examples: Dict[int, Tuple[np.ndarray, np.ndarray]],
) -> float:
    """Class-imbalance pos_weight, computed directly from the cached labels.

    Refactor note: this function used to call `active_hosts_in_window` and
    `host_label_at` for every host at every decision time, which made
    computing pos_weight comparable in cost to a whole training pass. Now
    we just sum integer labels across the precomputed cache: O(total examples)
    pure NumPy work and no per-host scanning.
    """
    pos = 0
    total = 0
    for _, labels in examples.values():
        pos += int(labels.sum())
        total += int(labels.size)
    neg = total - pos
    pos = max(pos, 1)
    return float(neg) / float(pos)


def train_one_epoch(model: TemporalHostRiskModel,
                    optimizer: torch.optim.Optimizer,
                    loss_fn: nn.Module,
                    mem_table_cpu: torch.Tensor,
                    history: List[deque],
                    decision_times_train: np.ndarray,
                    examples: Dict[int, Tuple[np.ndarray, np.ndarray]],
                    cfg: Config,
                    epoch: int,
                    device: torch.device,
                    ) -> Tuple[torch.Tensor, List[deque], float]:
    """One training epoch over the chronological train period.

    Refactor notes:
    - Active hosts and labels are read from the precomputed `examples`
      cache instead of being recomputed inside the loop.
    - `mem_table` lives on CPU; the explicit `mem_table.detach()` call from
      the previous version is no longer needed because the CPU table is a
      pure data buffer with no autograd graph.
    """
    model.train()
    losses_in_epoch: List[float] = []
    accum_loss = torch.zeros((), device=device)
    n_accum = 0
    optimizer.zero_grad()
    # Start at T_MIN - 1 so events with t == T_MIN are included in the first
    # replay window. (chronological_event_indices uses an exclusive lower bound.)
    prev_t = T_MIN - 1

    pbar = tqdm(
        enumerate(decision_times_train),
        total=len(decision_times_train),
        desc=f"Epoch {epoch}/{cfg.num_epochs} (decision times)",
        unit="dt",
        leave=False,
        dynamic_ncols=True,
    )
    for _, dt in pbar:
        dt = int(dt)
        # 1) Replay events (prev_t, dt]. We pass `update_with_grad=False`
        #    because the new CPU `mem_table` is written via `.detach()`,
        #    so there is no autograd path from the GRU/msg through `mem_table`
        #    back to the loss. Gradients still flow into the event encoder
        #    via `build_context_batch` (which re-encodes the context events
        #    with grad enabled), and into the attention + combine + risk
        #    head via the prediction. Skipping the replay-time graph saves
        #    a large amount of GPU memory on long replay windows.
        mem_table_cpu = replay_through_time(
            model, mem_table_cpu, history, prev_t, dt, cfg,
            update_with_grad=False, device=device,
        )
        # 2) Score active hosts at dt — read from precomputed cache
        ex = examples.get(dt)
        if ex is not None:
            hosts_np, labels_np = ex
            hosts_cpu = torch.from_numpy(hosts_np.astype(np.int64))
            label_t = torch.from_numpy(labels_np.astype(np.float32)).to(device)

            qm, cc, cf, cd, mk = build_context_batch(
                model, mem_table_cpu, history, hosts_cpu, dt, cfg, device,
            )
            logits = model.predict(qm, cc, cf, cd, mk)
            loss = loss_fn(logits, label_t)
            accum_loss = accum_loss + loss
            n_accum += 1

        # 3) Periodic backward; mem_table is CPU data so no detach is needed.
        if n_accum >= cfg.decisions_per_grad_step:
            (accum_loss / n_accum).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            step_loss = float(accum_loss.detach().item()) / n_accum
            losses_in_epoch.append(step_loss)
            pbar.set_postfix(loss=f"{step_loss:.4f}", grad_steps=len(losses_in_epoch))
            optimizer.zero_grad()
            accum_loss = torch.zeros((), device=device)
            n_accum = 0

        prev_t = dt

    if n_accum > 0:
        (accum_loss / n_accum).backward()
        optimizer.step()
        step_loss = float(accum_loss.detach().item()) / n_accum
        losses_in_epoch.append(step_loss)
        pbar.set_postfix(loss=f"{step_loss:.4f}", grad_steps=len(losses_in_epoch))

    avg = float(np.mean(losses_in_epoch)) if losses_in_epoch else float("nan")
    if cfg.verbose:
        print(f"[epoch {epoch}] train loss avg = {avg:.4f} over "
              f"{len(losses_in_epoch)} grad steps")
    return mem_table_cpu, history, avg

## 14. Validation and early stopping

Validation is a no-grad replay over the validation period that **continues from the end
of the training memory state**. We compute validation loss and AUROC; we early-stop on
non-improving validation AUROC for `early_stop_patience` epochs.

Crucially: we do NOT update `mem_table` from the validation/test data with gradients,
and we do NOT shuffle in time. This avoids leakage.

In [ ]:
from tqdm.auto import tqdm


@torch.no_grad()
def evaluate_split(model: TemporalHostRiskModel,
                   mem_table_cpu: torch.Tensor,
                   history: List[deque],
                   decision_times_eval: np.ndarray,
                   examples: Dict[int, Tuple[np.ndarray, np.ndarray]],
                   cfg: Config,
                   prev_t_init: int,
                   device: torch.device,
                   loss_fn: Optional[nn.Module] = None,
                   ) -> Tuple[torch.Tensor, List[deque], Dict[str, float],
                              np.ndarray, np.ndarray]:
    """Evaluate on a chronological split, replaying memory state without grad.

    Refactor notes:
    - Active hosts and labels are read from the `examples` cache instead of
      being recomputed every call.
    - The chronological memory state is preserved across calls just like
      before (we continue from `prev_t_init`); we only changed *how* we
      look up examples, not the temporal-leakage discipline.
    - Reads from the persistent device-side event tensors via
      `replay_through_time`; nothing here calls `.to(DEVICE)` on a full
      event tensor anymore.
    """
    model.eval()
    all_probs: List[float] = []
    all_labels: List[int] = []
    losses: List[float] = []
    prev_t = int(prev_t_init)

    eval_bar = tqdm(
        decision_times_eval,
        total=len(decision_times_eval),
        desc="Eval (decision times)",
        unit="dt",
        leave=False,
        dynamic_ncols=True,
    )
    for dt in eval_bar:
        dt = int(dt)
        mem_table_cpu = replay_through_time(
            model, mem_table_cpu, history, prev_t, dt, cfg,
            update_with_grad=False, device=device,
        )
        ex = examples.get(dt)
        if ex is None:
            prev_t = dt
            continue
        hosts_np, labels_np = ex
        hosts_cpu = torch.from_numpy(hosts_np.astype(np.int64))
        label_t = torch.from_numpy(labels_np.astype(np.float32)).to(device)

        qm, cc, cf, cd, mk = build_context_batch(
            model, mem_table_cpu, history, hosts_cpu, dt, cfg, device,
        )
        logits = model.predict(qm, cc, cf, cd, mk)
        if loss_fn is not None:
            batch_loss = float(loss_fn(logits, label_t).item())
            losses.append(batch_loss)
            eval_bar.set_postfix(loss=f"{batch_loss:.4f}")
        all_probs.extend(torch.sigmoid(logits).detach().cpu().tolist())
        all_labels.extend(int(x) for x in labels_np.tolist())
        prev_t = dt

    probs = np.asarray(all_probs, dtype=np.float64)
    labels_arr = np.asarray(all_labels, dtype=np.int64)
    metrics: Dict[str, float] = {
        "loss": float(np.mean(losses)) if losses else float("nan")
    }
    if labels_arr.size > 0 and 0 < labels_arr.sum() < labels_arr.size:
        metrics["auroc"] = float(roc_auc_score(labels_arr, probs))
        metrics["ap"] = float(average_precision_score(labels_arr, probs))
    else:
        metrics["auroc"] = float("nan")
        metrics["ap"] = float("nan")
    return mem_table_cpu, history, metrics, probs, labels_arr

### Sanity pass before the full training run

Before running the full training, we run a tiny end-to-end check: build the model,
forward a single batch of fabricated examples, and confirm shapes and loss are sensible.
This catches dimensional bugs in seconds rather than after a long training session.

In [ ]:
def build_model(cfg: Config) -> TemporalHostRiskModel:
    model = TemporalHostRiskModel(cat_vocab_sizes, num_numeric=event_num.size(-1), cfg=cfg).to(DEVICE)
    return model


# --- Sanity pass on a tiny synthetic batch ---
sanity_model = build_model(CFG)
sanity_model.eval()
with torch.no_grad():
    B = 4
    qm = torch.randn(B, CFG.memory_dim, device=DEVICE)
    cc = torch.randn(B, CFG.K_recent_events, CFG.memory_dim, device=DEVICE)
    cf = torch.randn(B, CFG.K_recent_events, CFG.event_repr_dim, device=DEVICE)
    cd = torch.rand(B, CFG.K_recent_events, device=DEVICE) * 3600.0
    mk = torch.ones(B, CFG.K_recent_events, dtype=torch.bool, device=DEVICE)
    logits = sanity_model.predict(qm, cc, cf, cd, mk)
    print("Sanity logits shape:", logits.shape, "(expected [B])")
    print("Sanity logits sample:", logits[:4].cpu().numpy())

# Encoder sanity
sanity_cat = {f: torch.zeros(2, dtype=torch.long, device=DEVICE) for f in CAT_FIELDS}
sanity_num = torch.zeros(2, event_num.size(-1), device=DEVICE)
sanity_evt = sanity_model.encode_events(sanity_cat, sanity_num)
print("Encoded event shape:", sanity_evt.shape, "(expected [2, event_repr_dim])")

### Full training with chronological splits and early stopping

We instantiate the model, compute a positive-class weight from the training period only,
and run the training loop. Per epoch:

1. Reset memory and history at the start.
2. Train across the train period.
3. Continue (no grad) into the validation period and compute metrics.
4. If validation AUROC improves, save the model. If it stops improving for
   `early_stop_patience` epochs, stop.

In [ ]:
from tqdm.auto import tqdm

# DEC_TRAIN / DEC_VAL / DEC_TEST and the per-split example dictionaries
# (train_examples, val_examples, test_examples) were precomputed in the
# "decision-example cache" cell above. We use them directly here.

print("Computing pos_weight from cached training labels (instant)...")
POS_WEIGHT = compute_pos_weight_from_split(train_examples)
print(f"pos_weight = {POS_WEIGHT:.2f}")

model = build_model(CFG)
optimizer = torch.optim.Adam(model.parameters(),
                             lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=DEVICE))

best_state = None
best_val_auroc = -float("inf")
patience_left = CFG.early_stop_patience
history_train_loss: List[float] = []
history_val_metric: List[Dict[str, float]] = []

epoch_loop = tqdm(range(1, CFG.num_epochs + 1), desc="Epochs", dynamic_ncols=True)
for epoch in epoch_loop:
    mem_table, history = reset_temporal_state(model, CFG)

    # Train across train period (consumes the cached train_examples)
    mem_table, history, train_loss = train_one_epoch(
        model, optimizer, loss_fn, mem_table, history,
        DEC_TRAIN, train_examples, CFG, epoch, DEVICE,
    )

    # Validation continues from the state after training, using val_examples
    mem_table, history, val_metrics, val_probs, val_labels = evaluate_split(
        model, mem_table, history, DEC_VAL, val_examples, CFG,
        prev_t_init=int(DEC_TRAIN[-1]) if len(DEC_TRAIN) > 0 else T_MIN - 1,
        device=DEVICE, loss_fn=loss_fn,
    )

    val_auroc = val_metrics.get("auroc", float("nan"))
    epoch_loop.set_postfix(
        train_loss=f"{train_loss:.4f}",
        val_auroc=f"{val_auroc:.4f}" if not math.isnan(val_auroc) else "nan",
        refresh=False,
    )

    history_train_loss.append(train_loss)
    history_val_metric.append(val_metrics)
    print(f"[epoch {epoch}] val: "
          + ", ".join(f"{k}={v:.4f}" for k, v in val_metrics.items()))

    if (not math.isnan(val_metrics.get("auroc", float("nan")))
            and val_metrics["auroc"] > best_val_auroc):
        best_val_auroc = val_metrics["auroc"]
        best_state = {k: v.detach().cpu().clone()
                      for k, v in model.state_dict().items()}
        patience_left = CFG.early_stop_patience
        print(f"  -> new best val AUROC = {best_val_auroc:.4f}, model saved")
    else:
        patience_left -= 1
        print(f"  -> no improvement; patience_left = {patience_left}")
        if patience_left <= 0:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict(best_state)
print(f"Best val AUROC: {best_val_auroc:.4f}")

## 15. Evaluation metrics and plots

We replay validation events first (to bring memory state up to the start of the test
period) and then evaluate the test period. Reported metrics:

- AUROC, average precision (PR AUC)
- Precision, recall, F1 at a configurable decision threshold (default 0.5)
- Confusion matrix
- ROC and PR curves
- Histogram of predicted probabilities (calibration sanity check)
- Training and validation loss curves
- Number of scored hosts per decision time

In [ ]:
# Replay through train+val (no grad) to land at the start of the test period.
# All three calls read from the corresponding precomputed example caches —
# we never recompute active hosts or labels here.
mem_table, history = reset_temporal_state(model, CFG)
mem_table, history, _, _, _ = evaluate_split(
    model, mem_table, history, DEC_TRAIN, train_examples, CFG,
    prev_t_init=T_MIN - 1, device=DEVICE, loss_fn=None,
)
mem_table, history, _, _, _ = evaluate_split(
    model, mem_table, history, DEC_VAL, val_examples, CFG,
    prev_t_init=int(DEC_TRAIN[-1]) if len(DEC_TRAIN) > 0 else T_MIN - 1,
    device=DEVICE, loss_fn=None,
)
mem_table, history, test_metrics, test_probs, test_labels = evaluate_split(
    model, mem_table, history, DEC_TEST, test_examples, CFG,
    prev_t_init=int(DEC_VAL[-1]) if len(DEC_VAL) > 0 else T_MIN - 1,
    device=DEVICE, loss_fn=loss_fn,
)
print("Test metrics:", test_metrics)


THRESHOLD = 0.5
y_pred_thresh = (test_probs >= THRESHOLD).astype(int)
if test_labels.sum() > 0:
    p, r, f1, _ = precision_recall_fscore_support(
        test_labels, y_pred_thresh, average="binary", zero_division=0
    )
    cm = confusion_matrix(test_labels, y_pred_thresh)
else:
    p = r = f1 = float("nan")
    cm = np.zeros((2, 2), dtype=int)
print(f"At threshold {THRESHOLD}: precision={p:.4f}, recall={r:.4f}, f1={f1:.4f}")
print("Confusion matrix [[TN, FP], [FN, TP]]:\n", cm)

In [ ]:
# --- Plots --------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# (a) training / validation curves
ax = axes[0, 0]
ax.plot(range(1, len(history_train_loss) + 1), history_train_loss,
        marker="o", label="train loss")
val_aurocs = [m.get("auroc", float("nan")) for m in history_val_metric]
val_losses = [m.get("loss", float("nan")) for m in history_val_metric]
ax.plot(range(1, len(val_losses) + 1), val_losses, marker="s", label="val loss")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(); ax.set_title("Loss curves")

# (b) val auroc curve
ax = axes[0, 1]
ax.plot(range(1, len(val_aurocs) + 1), val_aurocs, marker="o", color="darkgreen")
ax.set_ylim(0, 1); ax.set_xlabel("epoch"); ax.set_ylabel("val AUROC")
ax.set_title("Validation AUROC")

# (c) histogram of predicted probabilities (test)
ax = axes[0, 2]
if len(test_probs) > 0:
    ax.hist(test_probs[test_labels == 0], bins=30, alpha=0.6, label="negatives")
    if (test_labels == 1).any():
        ax.hist(test_probs[test_labels == 1], bins=30, alpha=0.8, label="positives", color="crimson")
ax.set_xlabel("predicted prob"); ax.set_ylabel("count"); ax.set_title("Test probabilities")
ax.legend()

# (d) ROC curve
ax = axes[1, 0]
if test_labels.sum() > 0 and test_labels.sum() < len(test_labels):
    fpr, tpr, _ = roc_curve(test_labels, test_probs)
    ax.plot(fpr, tpr)
    ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title(f"ROC (AUROC={test_metrics.get('auroc', float('nan')):.3f})")

# (e) PR curve
ax = axes[1, 1]
if test_labels.sum() > 0:
    pr, rc, _ = precision_recall_curve(test_labels, test_probs)
    ax.plot(rc, pr)
ax.set_xlabel("recall"); ax.set_ylabel("precision")
ax.set_title(f"Precision-Recall (AP={test_metrics.get('ap', float('nan')):.3f})")

# (f) class imbalance over decision times in TEST — reads from cache only
ax = axes[1, 2]
n_pos_per_dt: List[int] = []
n_hosts_per_dt: List[int] = []
for dt in DEC_TEST:
    ex = test_examples.get(int(dt))
    if ex is None:
        n_pos_per_dt.append(0); n_hosts_per_dt.append(0); continue
    hosts_np, labels_np = ex
    n_pos_per_dt.append(int(labels_np.sum()))
    n_hosts_per_dt.append(int(hosts_np.size))
ax.plot(DEC_TEST, n_hosts_per_dt, label="active hosts", color="steelblue")
ax.plot(DEC_TEST, n_pos_per_dt, label="positives", color="crimson")
ax.set_xlabel("decision time"); ax.set_ylabel("count")
ax.set_title("Test: scored hosts vs positives")
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(CFG.figures_dir, "training_eval_panel.png"), dpi=120)
plt.show()

## 16. Example predictions and interpretation

We pull out the highest-risk predictions in the test period and look at the host's
recent activity. This is the kind of artifact a defender would inspect.

In [ ]:
# Build a flat record (host, dt, prob, label) for the test split for inspection.
# This reads from the precomputed example caches (no per-host scanning).
records: List[Dict[str, float]] = []
mem_table, history = reset_temporal_state(model, CFG)
mem_table, history, _, _, _ = evaluate_split(
    model, mem_table, history, DEC_TRAIN, train_examples, CFG,
    prev_t_init=T_MIN - 1, device=DEVICE,
)
mem_table, history, _, _, _ = evaluate_split(
    model, mem_table, history, DEC_VAL, val_examples, CFG,
    prev_t_init=int(DEC_TRAIN[-1]) if len(DEC_TRAIN) > 0 else T_MIN - 1,
    device=DEVICE,
)

prev_t = int(DEC_VAL[-1]) if len(DEC_VAL) > 0 else T_MIN - 1
model.eval()
with torch.no_grad():
    for dt in DEC_TEST:
        dt = int(dt)
        mem_table = replay_through_time(
            model, mem_table, history, prev_t, dt, CFG,
            update_with_grad=False, device=DEVICE,
        )
        ex = test_examples.get(dt)
        if ex is None:
            prev_t = dt; continue
        hosts_np, labels_np = ex
        hosts_cpu = torch.from_numpy(hosts_np.astype(np.int64))
        qm, cc, cf, cd, mk = build_context_batch(
            model, mem_table, history, hosts_cpu, dt, CFG, DEVICE,
        )
        probs = torch.sigmoid(model.predict(qm, cc, cf, cd, mk)).cpu().numpy()
        for h, p, y in zip(hosts_np.tolist(), probs.tolist(), labels_np.tolist()):
            records.append({"dt": dt, "host_id": int(h), "host": id2host[int(h)],
                            "prob": float(p), "label": int(y)})
        prev_t = dt

results_df = pd.DataFrame(records).sort_values("prob", ascending=False)
print("Top-20 highest-risk predictions in test period:")
print(results_df.head(20))
print()
print("Top-10 confirmed-positive predictions ranked by prob:")
print(results_df[results_df["label"] == 1].head(10))

## Benchmarks: where is wall time going?

A small benchmark cell so you can see at a glance which stages dominate wall
time after the refactor. We separately time:

1. **loading cached numeric features** (warm cache) and a **recompute-from-scratch** comparison (using `force_recompute_features=True`),
2. **precomputing the decision-example cache** (warm and cold),
3. **one call to `replay_through_time`** on a short interval,
4. **one full training epoch** on the reduced debug config.

If your timings are dominated by stage (1) or (2), increase the cache hit
rate (don't toggle `force_recompute_*`). If stage (3) dominates, check
whether your `replay_encode_chunk_size` is too small. If stage (4) still
dominates after the others, that's the actual modeling work and is the
right thing to spend time on.

In [ ]:
# ---- Benchmark: per-stage wall-time so you can find the next bottleneck ----
import time
from contextlib import contextmanager


@contextmanager
def _timed(label: str):
    t0 = time.perf_counter()
    yield
    dt = time.perf_counter() - t0
    print(f"  {label:<55s} {dt:8.3f} s")


print("Benchmark (debug config):")

# (1) numeric features cache: warm load vs forced recompute
with _timed("(1a) load_or_compute_numeric_features (warm cache)"):
    _ = load_or_compute_numeric_features(events_df, CFG)

# Build a temporary CFG override so we don't permanently flip the user flag.
_cfg_force_feats = Config(**{**asdict(CFG), "force_recompute_features": True})
with _timed("(1b) compute_event_features_fast (cold, full recompute)"):
    _ = compute_event_features_fast(events_df, _cfg_force_feats)

# (2) decision-example cache: warm load vs cold recompute
with _timed("(2a) load_or_compute_decision_cache (warm cache)"):
    _ = load_or_compute_decision_cache(decision_times, CFG, n_events=len(t_arr))

with _timed("(2b) precompute_decision_examples (cold, in-memory)"):
    _ = precompute_decision_examples(decision_times, CFG)

# (3) one call to replay_through_time on a short interval
_bench_model = build_model(CFG)
_bench_mem, _bench_hist = reset_temporal_state(_bench_model, CFG)
_short_from = T_MIN - 1
_short_to = int(min(T_MAX, T_MIN + 30 * 60))  # first 30 minutes of events
with _timed(f"(3)  replay_through_time over [{_short_from}, {_short_to}]"):
    _bench_mem = replay_through_time(
        _bench_model, _bench_mem, _bench_hist,
        _short_from, _short_to, CFG, update_with_grad=False, device=DEVICE,
    )

# (4) one full training epoch on the debug config
_epoch_model = build_model(CFG)
_epoch_opt = torch.optim.Adam(_epoch_model.parameters(), lr=CFG.learning_rate,
                              weight_decay=CFG.weight_decay)
_epoch_pw = compute_pos_weight_from_split(train_examples)
_epoch_loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(_epoch_pw, device=DEVICE)
)
_epoch_mem, _epoch_hist = reset_temporal_state(_epoch_model, CFG)
with _timed("(4)  train_one_epoch on debug config"):
    _epoch_mem, _epoch_hist, _ = train_one_epoch(
        _epoch_model, _epoch_opt, _epoch_loss_fn,
        _epoch_mem, _epoch_hist, DEC_TRAIN, train_examples, CFG,
        epoch=0, device=DEVICE,
    )

print("Benchmark complete.")

## Implementation Notes and Design Choices

This section explicitly states the trade-offs we made, so a future reader can change them
without guessing.

**Why hosts are nodes (and not users).**
Lateral movement is fundamentally about traversal between machines. Users authenticate
*from* a host *to* another host; the same user account can act safely on one host and
suspiciously on another. Treating users as separate nodes inflates the graph and
distributes the signal across many low-volume nodes. Instead we treat the user identity
and domain as **event attributes** that the encoder embeds into each event vector.

**Why event-driven memory updates.**
Authentication arrives as an irregularly spaced stream. Bucketing into fixed time windows
loses ordering and exact timing within the window. By updating each host's memory with a
`GRUCell` per event, we preserve continuous-time semantics and let the model learn how
much to forget between events via the GRU gates. The sinusoidal time encoding feeds the
explicit gap into the message function.

**Why fixed decision times.**
Real defenders look at risk dashboards on a cadence (here, every 5 minutes). Predicting
at every event is operationally noisy and would score the same host hundreds of times in
quick succession. Fixed decision intervals also make the prediction task well-defined
for evaluation: a clear count of scored host-time pairs.

**Why event-based aggregation, not "k-hop neighbors".**
Many GNN tutorials aggregate over the static neighbors of a node. In a temporal graph
the natural unit is *recent interactions*, not *neighbors at some unspecified time*. We
gather the host's last `K` events and attend over them. This naturally weights repeated
interactions and respects time order via the time encoding.

**Why this is forecasting, not anomaly detection.**
We want to flag *upcoming* compromise, not just to detect already-anomalous patterns.
Labels live in the *future* relative to the decision time, so the model is rewarded for
predicting events it has not yet seen. Calibration and PR-AUC matter more than
classical anomaly scores.

**Approximation: counterpart memory at event time.**
At decision time, for each historical context event we want the *other* endpoint's
memory at the time the event occurred. The faithful implementation stores those
snapshots (one per endpoint per event) — that is exactly what we do via the deque's
`counterpart_mem` field, with `.detach()` so it is fixed data, not a parameter. The
alternative (using the counterpart's *current* memory at decision time) is cheaper but
loses temporal fidelity. We chose the faithful version because the per-host deque keeps
storage proportional to `N_HOSTS * K * memory_dim` rather than `O(events)`.

**Truncated BPTT and the CPU `mem_table`.**
The original notebook stored `mem_table` on GPU and relied on `.detach()` after each
optimizer step to bound the autograd graph. In the refactor we go further: the
`mem_table` lives on **CPU** and every endpoint write is `.detach().cpu()`. This was a
deliberate trade-off:

- *Pros:* large constant savings in GPU memory (the table is `N_HOSTS * memory_dim`
  and otherwise grows unused autograd state through replay), simpler reasoning about
  state, and the ability to use longer replay windows without OOM.
- *Cons:* gradients no longer flow from the loss back into the `HostMemoryModule`'s
  `GRUCell`/message MLP through the memory-update path. The encoder, attention,
  combine layer and risk head still receive gradients normally — gradients flow into
  the **event encoder** through the context-event re-encoding inside
  `build_context_batch`, and into the rest of the head from `model.predict`.

In practice this was a good trade for the demo config: the bulk of the model's
expressive power comes from the encoder + attention head, and the memory update acts
more like a learned online state estimator than a parameter we strictly need to fit
through long unrolled chains.

**Chunked replay encoding.**
`replay_through_time` processes events in chunks of `replay_encode_chunk_size`
(default 4096). Inside each chunk we batch-encode all events in one GPU forward pass,
then apply the GRU updates one event at a time to preserve the event ordering.
Without chunking, encoding all events in a long replay window in a single batch would
produce huge intermediate tensors and either OOM or thrash the GPU allocator.

**Cached decision examples.**
The single biggest speedup vs. the original notebook: `(active hosts, labels)` for
every decision time is computed **once** in preprocessing (with on-disk pickle cache),
not recomputed every epoch. Training and evaluation read from this cache by `dt`.
`compute_pos_weight_from_split` was rewritten to be O(total examples) on the cache
arrays rather than O(epochs * decision_times * hosts).

**Persistent device-side event tensors.**
`event_num_dev` and `event_cat_dev` are uploaded to the GPU **once** at preprocessing
and then re-used everywhere. The hot loops index those device tensors with device
index tensors — no `.to(DEVICE)` on a full event tensor inside the training loop.

**No PyTorch Geometric.**
PyG is excellent for static or batched graphs, but here the temporal structure is so
specific (event-by-event memory + per-host context windows) that wrapping it in PyG's
abstractions would add indirection without simplifying the code. A reader can swap the
attention layer for `torch_geometric.nn.TransformerConv` if they want to leverage PyG.

## 17. Notes on limitations and next steps

**Limitations (be honest):**

- We trained on a sampled prefix of the LANL log (`sample_nrows`). Behavior on the full
  58-day stream may differ; in particular, the rare red-team events become much rarer
  per decision time when scaled.
- Our "malicious involvement" labels are derived from red-team marks — they are an
  operational proxy, not a clean ground truth.
- `host_history` retains the *last* counterpart memory snapshot per event. We do not
  refresh those snapshots if the counterpart's memory drifts later. This is intentional
  (we want the *historical* state) but means the snapshots can age out.
- Class imbalance is severe at 5-minute resolution; AUROC can look optimistic even
  when PR-AUC is low.
- We do no negative subsampling: every active host at every decision time contributes
  to the loss. For very large enterprises, smarter negative sampling helps.
- We do not include process telemetry, DNS, or netflow — features that the literature
  shows are complementary signals for lateral movement.

**Next steps to push this further:**

- **Exact historical memory snapshots** stored as a tensor instead of approximations,
  with periodic compaction.
- **Heterogeneous host-user-process graph** with HeteroGNN message passing.
- **Smarter negative sampling**, e.g., positives + matched-prior-traffic negatives.
- **Streaming/online training** that does not require chronological replay each epoch.
- **Richer event features**: time of day, day of week, process ancestry, flow stats.
- **Self-supervised pretraining**: predict next interacting host given the recent
  history; fine-tune on the rare-event labels.
- **Calibration improvements**: temperature scaling on a held-out calibration block
  before deployment.
- **Multi-horizon forecasting**: predict for `H = {15min, 30min, 1hr}` jointly.

**Reproducibility:**
The full `Config` is printed near the top, model weights are saved in `artifacts_dir`,
figures are saved in `figures_dir`. Random seeds are set in `set_seed`.

In [ ]:
# Persist the trained model and the config for reproducibility.
torch.save({"state_dict": model.state_dict(),
            "cfg": asdict(CFG),
            "cat_vocab_sizes": cat_vocab_sizes,
            "num_numeric": event_num.size(-1)},
           os.path.join(CFG.artifacts_dir, "temporal_host_risk.pt"))
print("Saved model to", os.path.join(CFG.artifacts_dir, "temporal_host_risk.pt"))